# 🛠️ Workshop 4 — Building Cooperative LLM Agent Workflows
## Anti-pattern Detection · Code Smells · Technical Debt Resolution

**LLMA4SE 2026** — 2nd International Summer School on LLM-based Agents for Software Engineering
Day 3 · 15:00 – 18:00 · Instructors: **Karthik Shivashankar** (SINTEF Digital / UiO) & **Adela Nedisan Videsjorden** (UiO)

---

### What you will build today

| Part | You build | Time |
|---|---|---|
| **0** | Setup: the toolbox, your key, and what an API call really is | 15 min |
| **1** | 🕵️ **The Code Auditor** — one agent, four deterministic tools (radon · pylint · PyExamine · MLScent) | 50 min |
| **2** | 🤝 **The Refactoring Team** — Auditor → Refactorer ⇄ QA, a blackboard, verification gates, a failure lab | 60 min |
| **3** | 💳 **The Technical-Debt Pipeline** — classify, triage, resolve, report | 40 min |
| **4** | 🚀 **Production frameworks** — the same team in LangGraph, an autonomous Deep Agent, and a packaged CLI | bonus |

### Runtime

**No GPU needed.** `Runtime → Change runtime type → CPU` is fine — the brain lives behind the **OpenAI API**.
You need one thing: an **OpenAI API key**.

> 💡 **The one idea to take home:** *deterministic tools measure, the LLM interprets, and a gate decides.*
> Everything else in this notebook is plumbing around that sentence.

### How to read this notebook 🧭

This is a **hands-on** three hours, not a lecture. Run every cell top to bottom; the notebook keeps score of
your predictions and tells you whether you are on schedule.

| Marker | What it wants from you | Time |
|---|---|---|
| 🎯 **Predict** | edit a `MY_GUESS_…` line **before** running — the notebook grades you | 30 s |
| 🧪 **Try it** | change one thing, re-run, watch what breaks | 1–3 min |
| 💬 **Discuss** | turn to your neighbour; there is no single right answer | 1–2 min |
| ✍️ **Exercise** | write code in the workspace cell; solutions are one click away | 10–15 min |
| ⏱ **Pit stop** | are we on schedule? run it and find out | 5 s |
| 🧠 **Lesson** | the sentence to remember when you forget the code | — |

> 🆘 **If you fall behind:** every Part ends with a cell that saves its result. Skip the ✍️ exercise, run the
> Part's last cell, and you can still start the next Part. Nothing later depends on your exercise answers.
>
> 🤝 **Work in pairs if you can.** One person drives the keyboard, the other reads the "Under the hood"
> sections out loud. Swap at every Part boundary.

---
# Part 0 · Setup

## 0.1 · Install the toolbox (~2 min)

| Package | What it is | Why an agent needs it |
|---|---|---|
| `radon` | complexity & maintainability metrics | fast, deterministic **eyes** |
| `pylint` | classic rule-based linter | a second pair of eyes with different blind spots |
| `code-quality-analyzer` | **PyExamine** (MSR 2025) — 49 metrics, 3 levels | research-grade smell detection |
| `ml-code-smell-detector` | **MLScent** (CAIN 2025) — 76 ML anti-pattern detectors | smells that only exist in ML code |
| `pytest` | test runner | the **QA gate** — the only reason we can trust an agent's patch |
| `pandas` | dataframes | scoring the debt classifier in Part 3 |
| `openai` | OpenAI SDK | the agent's **brain** |
| `langgraph`, `deepagents` | agent frameworks | Part 4 — the production rebuild |

In [ ]:
%pip install -q code-quality-analyzer ml-code-smell-detector radon pylint pytest pandas langgraph deepagents openai
print("✅ toolbox installed")
print("   ⏳ Colab may say 'restart session' — you can safely IGNORE it for this notebook.")

## 0.2 · Your OpenAI API key 🔑

Three ways to provide it — the loader below tries them **in order** and stops at the first that works.

**① Colab Secrets** *(recommended — the key never lands in the notebook file)*
> Click the **🔑 icon** in Colab's left sidebar → **+ Add new secret** → name it `OPENAI_API_KEY`,
> paste the value → flip **Notebook access** on. Optionally add a second secret `OPENAI_MODEL`
> (e.g. `gpt-4.1-mini`) to pin the model.

**② A `.env` file** — Colab left sidebar → 📁 **Files** → **Upload**, a file called `.env` containing:
```
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4.1-mini
```

**③ Typed prompt** — a hidden `getpass` box appears if neither of the above is found. Fine for a workshop,
but you will have to retype it every time the runtime restarts.

> ⚠️ **Never paste a key into a code cell.** Notebooks get shared, committed and screenshotted. This is not
> paranoia — it is the #1 way keys leak.

> 🧠 **Why this matters beyond today:** the key travels through the **environment**, never through the code.
> That single habit is what lets the exact same notebook, package and CI job run with a different key in each
> place, with no edits.

In [ ]:
import os, getpass, pathlib

IN_COLAB = pathlib.Path("/content").exists()


def load_dotenv() -> None:
    """Copy every KEY=value line of a .env into os.environ. ~8 lines beat a dependency."""
    for candidate in (pathlib.Path("/content/.env"), pathlib.Path(".env")):
        if not candidate.exists():
            continue
        for line in candidate.read_text().splitlines():
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            name, value = line.split("=", 1)
            os.environ.setdefault(name.strip(), value.strip().strip("'\""))
        print(f"📄 read {candidate}")


def load_openai_key() -> str:
    """.env  →  Colab secret  →  typed prompt. First hit wins."""
    load_dotenv()                       # runs first, so OPENAI_MODEL is picked up too

    if os.environ.get("OPENAI_API_KEY"):
        print("🔑 key found in the environment")
        return os.environ["OPENAI_API_KEY"]

    # Colab Secrets — the recommended route: the key never enters the notebook file
    try:
        from google.colab import userdata
        for name in ("OPENAI_API_KEY", "OPENAI_MODEL"):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass                    # OPENAI_MODEL is optional
        if os.environ.get("OPENAI_API_KEY"):
            print("🔑 key loaded from Colab Secrets")
            return os.environ["OPENAI_API_KEY"]
    except Exception:
        pass

    # 3) ask
    key = getpass.getpass("Paste your OpenAI API key (input hidden): ").strip()
    os.environ["OPENAI_API_KEY"] = key
    print("🔑 key set for this session")
    return key

_ = load_openai_key()
print("✅ key ends with …" + _[-4:] if _ else "❌ no key!")
print("🌍 environment:", "Google Colab" if IN_COLAB else "local Jupyter")
print("   Every later cell — including the `!debtbuster` shell commands in Part 4 — inherits this")
print("   key from os.environ. Set it once here; nothing else in the notebook ever sees it.")

## 0.3 · The brain: one `llm()` function

Every agent in this notebook — all six of them — calls **this one function**. That is deliberate: swapping the
model, the provider or the temperature is a one-line change, and the rest of the workshop cannot tell
the difference. It also absorbs the one API wrinkle you will hit in practice: **reasoning models**
(`gpt-5.x`, `o*`) reject `temperature` and rename `max_tokens`.

We also keep a **cost meter**, because "how much did my agent just spend?" is a production question, not an afterthought.

In [ ]:
import os
from openai import OpenAI

client = OpenAI()                     # reads OPENAI_API_KEY from the environment

# Your .env may pin a model with OPENAI_MODEL=...; otherwise this cheap, capable default is used.
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
#   "gpt-4.1-mini"  · good code model, cheap, fast          ← recommended for a live workshop
#   "gpt-4o-mini"   · cheapest, weakest at long rewrites
#   "gpt-4.1"       · strongest classic model here
#   "gpt-5.x" / "o*" · reasoning models: slower, no temperature knob, but the best refactorers

# Reasoning models rename max_tokens and reject temperature. Detect once, adapt everywhere.
IS_REASONING = MODEL_NAME.startswith(("gpt-5", "o1", "o3", "o4"))
print(f"🧠 brain: {MODEL_NAME}" + ("  (reasoning model)" if IS_REASONING else ""))

USAGE = {"calls": 0, "in": 0, "out": 0}          # the cost meter


def llm(user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        max_new_tokens: int = 1024,
        temperature: float = 0.2,
        json_mode: bool = False) -> str:
    """One call to the OpenAI API. EVERY agent in this workshop goes through here."""
    kwargs = dict(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
    )
    if IS_REASONING:
        # reasoning tokens are billed out of the same budget, so give it room to think
        kwargs["max_completion_tokens"] = max(max_new_tokens * 4, 4000)
    else:
        kwargs["max_tokens"] = max_new_tokens
        kwargs["temperature"] = temperature
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    try:
        resp = client.chat.completions.create(**kwargs)
    except Exception as e:                       # last-ditch adaptation for unknown models
        print(f"   ↻ retrying without model-specific params ({type(e).__name__})")
        kwargs.pop("temperature", None)
        if "max_tokens" in kwargs:
            kwargs["max_completion_tokens"] = max(kwargs.pop("max_tokens") * 4, 4000)
        resp = client.chat.completions.create(**kwargs)

    USAGE["calls"] += 1
    USAGE["in"] += resp.usage.prompt_tokens
    USAGE["out"] += resp.usage.completion_tokens
    return (resp.choices[0].message.content or "").strip()


def cost_report(price_in: float = 0.40, price_out: float = 1.60):
    """Rough $ estimate. Defaults are gpt-4.1-mini list prices, $ per 1M tokens."""
    usd = USAGE["in"] / 1e6 * price_in + USAGE["out"] / 1e6 * price_out
    print(f"📊 {USAGE['calls']} calls · {USAGE['in']:,} in / {USAGE['out']:,} out tokens · ≈ ${usd:.4f}")


# Smoke test — is the brain alive?
print(llm("In one sentence: what is a code smell?"))
cost_report()

> 🧪 **Try it (30 s):** re-run the cell with `temperature=1.5`. Then `0.0`. Which setting do you want for a
> *refactoring* agent, and why? (Hold that thought — Part 2 depends on the answer.)

## 0.4 · Under the hood: what an API call *actually* is

Before you build agents out of `llm()`, spend three minutes on what that function really does. Almost every
agent bug you will hit today is one of these four facts biting you.

**1 · There is no conversation.** The API is *stateless*. Every call sends the whole context again; the model
remembers nothing between calls. What looks like memory in a chatbot is the client resending the transcript.
Our agents exploit this: each one gets a clean, purpose-built context with no contamination from the last agent.

**2 · A "message" is just a labelled string.** You send a list of `{"role", "content"}` dicts. `system` sets
the rules, `user` carries the task. The model sees them concatenated with special separator tokens — the roles
are a *convention the model was trained on*, not an enforcement mechanism. This is exactly why a system prompt
can be ignored, and why we never rely on it alone for anything that matters (hence: gates).

**3 · Everything is tokens.** Text is split into sub-word tokens (~4 characters each in English, fewer in code).
You pay for input tokens *and* output tokens, at different rates. `max_tokens` caps the **output** only — and
if the model hits that cap mid-function, you get **truncated code**. That is failure pattern #2, and it is why
gate 1 (`ast.parse`) exists.

**4 · Generation is sampling, not lookup.** The model produces a probability distribution over the next token
and *samples* from it. `temperature` reshapes that distribution: `0.0` always takes the most likely token
(near-deterministic), `1.0+` flattens it (creative, erratic). For refactoring we want boring and repeatable,
so every agent today runs at `0.0–0.2`.

> ⚠️ **"Near-deterministic", not deterministic.** Even at `temperature=0` the same prompt can give different
> answers — floating-point non-determinism in batched GPU inference, and model updates behind the same name.
> **Never build a pipeline that assumes byte-identical LLM output.** Build gates instead.

Let's watch all four facts at once.

In [ ]:
# ── Fact 3: what tokens look like, and what they cost ──────────────
probe = "def process_order(self, order):"
r = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": f"Repeat this exactly: {probe}"}],
    **({"max_completion_tokens": 4000} if IS_REASONING else {"max_tokens": 50}),
)
u = r.usage
print(f"prompt sent      : {len(probe)} characters")
print(f"tokens billed IN : {u.prompt_tokens}   ← the whole messages list, not just your string")
print(f"tokens billed OUT: {u.completion_tokens}")
print(f"reply            : {r.choices[0].message.content!r}")
print("\nRule of thumb: ~4 chars per token of English, ~3 for code.")
print(f"Our smelly module is {len(open('/dev/null').read()) if False else 2906} chars ≈ 900 tokens —")
print("and the refactorer sends it TWICE (source + its own rewrite). Context adds up fast.")

In [ ]:
# ── Fact 4: temperature, seen directly ─────────────────────────────
# Reasoning models fix temperature at 1, so this experiment only runs on classic models.
if IS_REASONING:
    print(f"⏭  {MODEL_NAME} is a reasoning model — temperature is locked. Skipping.")
    print("   Set MODEL_NAME = 'gpt-4.1-mini' in §0.3 to run this experiment.")
else:
    q = "Name one Python code smell. Answer with just the name."
    for temp in (0.0, 0.0, 1.6, 1.6):
        print(f"  temp={temp}: {llm(q, temperature=temp, max_new_tokens=12)!r}")
    print("\n↑ The two temp=0.0 rows usually match. The two temp=1.6 rows usually don't.")
    print("  A refactoring agent at 1.6 would invent a different API every retry.")

> 💬 **Discuss (1 min):** the QA gate re-runs the *tests*, never the *model*. Given fact 4, why is that the only
> defensible design? *(Because you cannot diff against a previous run you cannot reproduce. The tests are the
> fixed point; the model output is the variable.)*

### The one-function rule

Notice what §0.3 did: **every** agent today goes through a single `llm()`. That is not tidiness, it is leverage:

| You want to… | Where you change it | Lines touched |
|---|---|---|
| swap model or provider | `MODEL_NAME` | 1 |
| add retries / rate-limit backoff | inside `llm()` | ~5 |
| log every prompt for debugging | inside `llm()` | 2 |
| enforce a global token budget | inside `llm()` | 3 |
| cache repeated calls | inside `llm()` | 3 |

A codebase that calls `client.chat.completions.create` in fifteen places can do none of these cheaply. This is
ordinary software engineering — it just matters more when every call costs money and can fail.

## 0.5 · A clean workspace, a clock and a scoreboard

Two housekeeping jobs in one cell.

**A working directory.** Everything the agents create lives in `/content/workshop` on the Colab VM (or
`./workshop` if you are running locally), so the static analysers never wander into Colab's `sample_data/`.

**Three tiny helpers that make the next three hours self-guiding:**

- `pit_stop("Part 1")` — am I on schedule? Run it at every Part boundary.
- `quiz(...)` — the 🎯 **Predict** cells call this to grade the guess you typed.
- `scoreboard()` — how did your intuition do against the machine? Printed at the end.

They are twenty lines of stdlib. No widgets, no dependencies, nothing to install — and they work identically
in Colab and in local Jupyter.

In [ ]:
import os, pathlib, sys, time, json, re, ast, subprocess

# The tools from §0.1 must be reachable as SHELL commands (radon, pylint, debtbuster),
# not just as imports. Colab already puts them on PATH; a local venv does not.
os.environ["PATH"] = str(pathlib.Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

WORKDIR = (pathlib.Path("/content/workshop") if pathlib.Path("/content").exists()
           else pathlib.Path("./workshop")).resolve()
WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)
(WORKDIR / "ml_project").mkdir(exist_ok=True)
print("📁 working in", os.getcwd())

# ── the clock ────────────────────────────────────────────────────────
T0 = time.time()
SCHEDULE = {"Part 0": 15, "Part 1": 65, "Part 2": 125, "Part 3": 165, "Part 4": 180}


def pit_stop(part: str) -> None:
    """⏱ Are we on schedule? Run at the end of each Part."""
    mins = (time.time() - T0) / 60
    drift = SCHEDULE[part] - mins
    mood = ("🟢 comfortably ahead — do the exercise" if drift > 5 else
            "🟡 right on time"                        if drift > -5 else
            "🟠 behind — skip the ✍️ exercise, run the Part's last cell, move on")
    print(f"⏱  {mins:5.0f} min elapsed │ {part} was budgeted to {SCHEDULE[part]} min │ {drift:+.0f} min · {mood}")


# ── the scoreboard ───────────────────────────────────────────────────
SCORE = {"asked": 0, "right": 0}


def quiz(question: str, your_answer, correct, explain: str = "") -> None:
    """🎯 Predict-then-reveal. `correct` may be a value or a predicate function."""
    ok = bool(correct(your_answer)) if callable(correct) else your_answer == correct
    SCORE["asked"] += 1
    SCORE["right"] += ok
    print("\n" + "─" * 72)
    print(f"🎯 {question}")
    print(f"   you said : {your_answer!r}   →   {'✅ nice call' if ok else '❌ not quite'}")
    if explain:
        print(f"   💡 {explain}")
    print(f"   🏅 running score: {SCORE['right']}/{SCORE['asked']}")
    print("─" * 72)


def scoreboard() -> None:
    n, r = SCORE["asked"], SCORE["right"]
    if not n:
        return print("🏅 no predictions made — you skipped the 🎯 cells!")
    verdict = ("you have good instincts about LLMs" if r / n >= 0.75 else
               "your instincts are being recalibrated — that is the point"
               if r / n >= 0.4 else "the machine surprised you today. Good.")
    print(f"🏅 prediction scoreboard: {r}/{n} — {verdict}")


# ── the JSON seatbelt ────────────────────────────────────────────────
def extract_json(text: str):
    """LLMs love to wrap JSON in chatter — dig the object/array back out.

    Used from §1.2 onward. Defined here because EVERY agent that returns
    structured output goes through it, and none of them may trust the model
    to have obeyed 'reply only with JSON'.
    """
    m = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError(f"No JSON found in: {text[:200]}")
    data = json.loads(m.group(0))
    return data.get("findings", data) if isinstance(data, dict) else data


pit_stop("Part 0")

---
# Part 1 · 🕵️ The Code Auditor Agent

**⏱ ~50 min**

One agent. Four tools. One JSON contract. By the end of this part you will have an agent that finds real
smells in real code and hands them to the next agent in a machine-readable form.

## 1.1 · Meet the patient 🤒

`inventory.py` is a small order-processing module that **works perfectly** — every test passes — yet it is
riddled with classic smells:

mutable default argument · dead code · long parameter list · long method · deep nesting · magic numbers · duplicated logic

> 🎯 **Look before you scroll:** give yourself 60 seconds to spot three of them by eye. That is the baseline
> your agent has to beat.

In [ ]:
%%writefile inventory.py
"""inventory.py -- Order processing for a small e-commerce shop.

This module works correctly (all tests pass!) but it is deliberately
full of code smells. Your agents will find and fix them.
"""

def helper_unused(x):          # SMELL: dead code -- never called anywhere
    return x * 2


class InventoryManager:
    """Manages stock and processes customer orders."""

    def __init__(self, items=[]):              # SMELL: mutable default argument
        self.items = {}
        for name, price, qty in items:
            self.items[name] = {"price": price, "qty": qty}
        self.log = []

    def add_item(self, name, price, qty, category, supplier, discount, taxable):
        # SMELL: long parameter list (7 params, most unused)
        self.items[name] = {"price": price, "qty": qty}
        return True

    def process_order(self, order):
        # SMELL: long method, deep nesting, magic numbers, duplication
        total = 0.0
        status = "ok"
        for name, qty in order:
            if name in self.items:
                if self.items[name]["qty"] >= qty:
                    if qty > 0:
                        price = self.items[name]["price"]
                        subtotal = price * qty
                        if subtotal > 100:                      # magic number
                            subtotal = subtotal - subtotal * 0.05   # magic number
                        if qty > 10:                            # magic number
                            subtotal = subtotal - subtotal * 0.02   # magic number
                        total = total + subtotal
                        self.items[name]["qty"] = self.items[name]["qty"] - qty
                        self.log.append("sold " + name)
                    else:
                        status = "invalid_qty"
                else:
                    status = "insufficient_stock"
            else:
                status = "unknown_item"
        total = total + total * 0.25            # magic number (VAT)
        return {"total": round(total, 2), "status": status}

    def refund_order(self, order):
        # SMELL: duplicated logic (mirror of process_order maths)
        total = 0.0
        for name, qty in order:
            if name in self.items:
                price = self.items[name]["price"]
                subtotal = price * qty
                if subtotal > 100:                              # magic number again
                    subtotal = subtotal - subtotal * 0.05
                if qty > 10:
                    subtotal = subtotal - subtotal * 0.02
                total = total + subtotal
                self.items[name]["qty"] = self.items[name]["qty"] + qty
        total = total + total * 0.25
        return {"total": round(total, 2), "status": "refunded"}

    def get_stock(self, name):
        if name in self.items:
            return self.items[name]["qty"]
        return 0

### The safety net 🥅

Before we let *any* AI touch this code, we pin down its behaviour with tests. These tests are the **contract**:
a refactoring is valid only if they stay green.

> 💬 **Discuss (30 s with your neighbour):** why must the tests exist *before* the refactoring agent runs,
> and not be written by the agent afterwards?

In [ ]:
%%writefile test_inventory.py
"""test_inventory.py -- Behaviour-preserving safety net.

These tests define the PUBLIC CONTRACT of the module. Any refactoring
your agents perform MUST keep every one of these green.
"""
import pytest
from inventory import InventoryManager


@pytest.fixture
def mgr():
    return InventoryManager([("widget", 10.0, 100), ("gizmo", 25.0, 5)])


def test_simple_order(mgr):
    result = mgr.process_order([("widget", 2)])
    assert result["status"] == "ok"
    assert result["total"] == 25.0          # 20 + 25% VAT


def test_bulk_discount_applied(mgr):
    # 20 widgets = 200 -> -5% (>100) -> -2% (>10 units) -> +25% VAT
    result = mgr.process_order([("widget", 20)])
    assert result["total"] == 232.75


def test_stock_is_decremented(mgr):
    mgr.process_order([("widget", 2)])
    assert mgr.get_stock("widget") == 98


def test_insufficient_stock(mgr):
    result = mgr.process_order([("gizmo", 99)])
    assert result["status"] == "insufficient_stock"


def test_unknown_item(mgr):
    result = mgr.process_order([("nonexistent", 1)])
    assert result["status"] == "unknown_item"


def test_refund_restores_stock(mgr):
    mgr.process_order([("widget", 2)])
    mgr.refund_order([("widget", 2)])
    assert mgr.get_stock("widget") == 100


def test_no_shared_state_between_instances():
    a = InventoryManager()
    b = InventoryManager()
    a.items["x"] = {"price": 1, "qty": 1}
    assert "x" not in b.items or a.items is not b.items

In [ ]:
# The smelly code WORKS — that's the whole point:
!python -m pytest test_inventory.py -q

## 1.2 · The experiment that justifies this whole workshop

Design Rule #1 says: *never make an LLM guess what a tool can measure.* That is easy to nod along to and easy
to forget the moment a model sounds confident. So let's **measure the failure** instead of asserting it.

We ask the model to compute cyclomatic complexity — a precisely defined, mechanically checkable number — with
**no tools**. Then we let `radon` compute the truth.

> 🎯 **Predict first — edit the cell before running it.** `inventory.py` has **6** functions and methods.
> Change `MY_GUESS_WRONG` on the first line to how many of the model's six complexity numbers you think will
> be **wrong**. Then run. The notebook grades you (±1 counts as a hit) and keeps score all afternoon.
>
> 🎤 **Show of hands first:** who guessed 0–1? 2–4? 5–7? Remember which camp you were in.

In [ ]:
MY_GUESS_WRONG = 3      # 🎯 EDIT ME (0–6) before running: how many will the model get wrong?

NO_TOOLS = """You are a static analysis engine. Compute the cyclomatic complexity
of EVERY function and method in the module. Reply ONLY with a JSON object mapping
"name" -> integer complexity. No prose, no explanation."""

guessed = extract_json(llm(f"```python\n{open('inventory.py').read()}\n```",
                           system_prompt=NO_TOOLS, max_new_tokens=400,
                           temperature=0.0, json_mode=True))

# The model tends to answer "InventoryManager.process_order"; radon says "process_order".
# Strip the qualifier so we compare NUMBERS, not naming conventions. (Keep both, though —
# that mismatch is failure pattern #2, "contract drift", and Part 2 will name it.)
guessed = {k.split(".")[-1]: v for k, v in guessed.items()}

radon_json = json.loads(subprocess.run(["radon", "cc", "-j", "inventory.py"],
                                       capture_output=True, text=True).stdout)
truth = {b["name"]: b["complexity"]                       # functions & methods only —
         for blocks in radon_json.values() for b in blocks # we never asked it about classes
         if b["type"] != "class"}

print(f"{'FUNCTION':22} {'LLM GUESS':>10} {'RADON (truth)':>14}   ")
print("-" * 60)
wrong = 0
for name, actual in sorted(truth.items()):
    g = guessed.get(name, "— (not answered)")
    ok = (g == actual)
    wrong += not ok
    print(f"{name:22} {str(g):>10} {actual:>14}   {'✅' if ok else '❌'}")
print("-" * 60)
print(f"{wrong}/{len(truth)} wrong — and every single answer was delivered with total confidence.")
print(f"Look at WHICH one it missed: {', '.join(n for n, a in truth.items() if guessed.get(n) != a) or '(none)'}.")

quiz(f"You predicted {MY_GUESS_WRONG} of {len(truth)} would be wrong. Reality: {wrong}.",
     MY_GUESS_WRONG, lambda g: abs(g - wrong) <= 1,
     "Close is not correct. The model is usually off by one on exactly the function that matters most — "
     "the deeply nested one — and it says so with no error, no warning, no uncertainty flag.")

### Read that table again

Two things are true at once, and holding both is the entire skill of this workshop:

1. **The model is bad at this — in the most dangerous way.** It is not wildly wrong; it is *nearly* right.
   Counting branch paths is arithmetic over a parse tree: models approximate, they do not execute. So most
   numbers land, and then one is off by one — usually on the gnarliest function, precisely where you needed
   the number. And it fails *silently*: no error, no uncertainty flag, just a wrong integer that looks exactly
   like a right one. A metric you cannot trust to ±1 is a metric you cannot gate on.

   > 🔍 **Did you notice the second failure?** The model answered `"InventoryManager.process_order"`;
   > `radon` says `"process_order"`. Nobody was wrong — the two just disagreed about the *format*, and our
   > first draft of this cell scored every method as "not answered". That is **contract drift**, failure
   > pattern #2 in Part 2, and it is the single most common reason agent pipelines break in production.
   > Fixing it took one line (`k.split(".")[-1]`). Finding it took an afternoon.
2. **The model is excellent at the thing radon cannot do.** `radon` will tell you `process_order` scores 7.
   It will never tell you *that the nesting exists because stock-checking, quantity-validation and pricing were
   never separated*, nor propose the extraction. That is judgement, and it is genuinely hard.

> ⚖️ **The division of labour, stated precisely:**
> **Tools produce facts. The LLM produces interpretations of facts. Gates decide whether to act on them.**
> Every agent you build today is an arrangement of those three roles — and every agentic system that fails in
> production has blurred two of them together.

This also tells you where to look when an agent misbehaves: if it got a *number* wrong, you gave it a job that
belonged to a tool.

## 1.3 · Deterministic tools first — the agent's "eyes" 👀

A core design rule of agentic software engineering:

> ⚖️ **Never make an LLM guess what a deterministic tool can measure.**
> Static analysers are fast, cheap, and never hallucinate. The LLM's job is *interpretation and action*, not *measurement*.

### Tool 1 — `radon`: complexity metrics

In [ ]:
import subprocess, json, pathlib

def run_radon(path: str) -> str:
    """Cyclomatic complexity (CC) per function + Maintainability Index (MI)."""
    cc = subprocess.run(["radon", "cc", "-s", path], capture_output=True, text=True).stdout
    mi = subprocess.run(["radon", "mi", "-s", path], capture_output=True, text=True).stdout
    return f"CYCLOMATIC COMPLEXITY (A=best, F=worst):\n{cc}\nMAINTAINABILITY INDEX (100=best):\n{mi}"

print(run_radon("inventory.py"))

📊 **Read the output:** `process_order` should stand out. CC counts independent paths through a function —
every `if` adds one. High CC = hard to test, hard to change.

### Tool 2 — `pylint`: rule-based linting

In [ ]:
def run_pylint(path: str, max_findings: int = 15) -> str:
    """Classic linter findings as compact text."""
    raw = subprocess.run(
        ["pylint", path, "--output-format=json", "--disable=C0114,C0115,C0116"],
        capture_output=True, text=True
    ).stdout
    try:
        issues = json.loads(raw)[:max_findings]
    except json.JSONDecodeError:
        return raw[:1500]
    return "\n".join(f"L{i['line']}: [{i['symbol']}] {i['message']}" for i in issues) or "no findings"

print(run_pylint("inventory.py"))

👀 Notice pylint catches the **mutable default argument** (`dangerous-default-value`) — a bug-in-waiting that
radon's metrics are blind to. Different tools see different smells. That is why real auditors combine them.

### Tool 3 — **PyExamine** 🔬 (research tool · MSR 2025)

PyExamine analyses code at **three levels** — code, structural, architectural — across 49 metrics.
`pip install code-quality-analyzer`

In [ ]:
def run_pyexamine(directory: str = ".") -> str:
    """PyExamine: multi-level smell detection (Shivashankar & Martini, MSR 2025).
    Findings are written to <output>.txt, so we run the CLI then read the report."""
    subprocess.run(
        ["analyze_code_quality", directory, "--type", "code",
         "--output", "pyexamine_report",
         "--ignore", "sample_data", ".config", "__pycache__"],
        capture_output=True, text=True, timeout=600,
    )
    report = pathlib.Path("pyexamine_report.txt")
    return report.read_text()[:3000] if report.exists() else "PyExamine produced no report"

print(run_pyexamine("."))

> 🧪 **Try it (2 min):** re-run with `--type structural`. Which *new* smells appear that the code-level pass missed?

### Tool 4 — **MLScent** 🤖 (smells that exist only in ML code · CAIN 2025)

Classic tools are blind to a whole category of rot that lives *only* in machine-learning code: NaN comparisons,
missing `zero_grad()`, unseeded randomness, data leakage. MLScent ships **76 detectors** for
PyTorch / TensorFlow / sklearn / pandas / numpy / HuggingFace.
`pip install ml-code-smell-detector`

In [ ]:
%%writefile ml_project/train_model.py
"""train_model.py -- Churn-prediction training script.

It trains fine... but is it reproducible? Is it healthy ML code?
Your ML Auditor agent (powered by MLScent) will tell you.
"""
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def load_data(path):
    df = pd.read_csv(path)
    for i in range(len(df)):                      # pandas: unnecessary iteration
        if df["age"][i] == np.nan:                # numpy: NaN equality (always False!)
            df["age"][i] = 0                      # pandas: chain indexing
    return df


def train():
    df = load_data("churn.csv")
    X = df.drop("label", axis=1).values
    y = df["label"].values
    # sklearn: no feature scaling, no pipeline, no random_state
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

    model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.003)   # hardcoded hyperparams
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):                      # no early stopping, no checkpoints
        out = model(torch.tensor(X_train, dtype=torch.float32))
        loss = loss_fn(out, torch.tensor(y_train))
        loss.backward()                           # pytorch: missing opt.zero_grad()
        opt.step()

    preds = model(torch.tensor(X_test, dtype=torch.float32)).argmax(1).numpy()
    print("accuracy:", accuracy_score(y_test, preds))   # over-reliance on accuracy
    # no torch.manual_seed / np.random.seed anywhere -> unreproducible


if __name__ == "__main__":
    train()

> 😱 Before running the detector: **spot 3 smells yourself** (60 seconds, no scrolling back).

Now wrap MLScent as a tool, exactly like the others. Note it never *runs* the script — it is pure AST analysis,
so it is safe on code you have never seen.

In [ ]:
def run_mlscent(project_dir: str = "ml_project") -> str:
    """MLScent: 76 ML-specific anti-pattern detectors (Shivashankar, CAIN 2025).
    Findings land in output/analysis_report.txt, so we run the CLI then read it."""
    subprocess.run(["ml_smell_detector", "analyze", project_dir],
                   capture_output=True, text=True, timeout=600)
    report = pathlib.Path("output/analysis_report.txt")
    return report.read_text()[:3500] if report.exists() else "MLScent produced no report"

print(run_mlscent("ml_project"))

📊 **Read the output:** MLScent flags the `== np.nan` bug (a *correctness* problem — that branch **never**
executes) and the missing `zero_grad()` (gradients accumulate across epochs → wrong training). Note each finding
ships a **"How to fix"** — that is structured, actionable input for an LLM, not just a complaint.

## 1.4 · Under the hood: the AST — how tools *see* code

Every deterministic tool today — radon, pylint, PyExamine, and the exercise you are about to write — works the
same way: it stops treating your file as **text** and starts treating it as a **tree**. That tree is the
Abstract Syntax Tree, and understanding it is the difference between grepping for `if` and actually analysing
code.

### From characters to a tree

Take one line:

```python
if qty > 10 and price < 5:
    total = total * 0.9
```

Three stages turn it into something a program can reason about:

1. **Tokenise** — chop the characters into words: `if`, `qty`, `>`, `10`, `and`, … A token knows it is a name
   or a number; it knows nothing about structure.
2. **Parse** — apply the grammar of Python to nest those tokens into a tree. Now `>` *has two operands*, and
   the assignment *is inside* the `if`.
3. **Walk** — visit the nodes and compute something.

The result is this shape (`ast.dump` prints it verbatim; here it is trimmed to the essentials):

```
If                                    ← "a branch happens here"
├── test: BoolOp(op=And)              ← "two conditions, short-circuited"
│   ├── Compare(qty > 10)
│   └── Compare(price < 5)
└── body:
    └── Assign(total = BinOp(total * 0.9))
```

**"Abstract" means the noise is gone.** No parentheses, no whitespace, no comments, no `# type:` hints about
formatting — those were needed to *write* the code, not to *describe* it. `if x>10:` and

```python
if (
    x > 10
):
```

produce the **identical** tree. That is the whole point: a tool built on the AST cannot be fooled by
formatting, and it cannot mistake the word `if` inside a string literal for a branch. Both are failure modes
of every regex-based "linter" ever written.

### Why this makes tools trustworthy

The tree is what lets a tool be *exact* rather than *approximately right*:

| Question | On text (regex) | On the AST |
|---|---|---|
| How many branches in this function? | count `if` — wrong on strings, comments, `elif` | count `If`/`For`/`While` nodes — exact |
| Is this parameter unused? | hopeless | walk the function body for `Name` nodes — exact |
| Is `0.25` a magic number? | matches `v0.25.1` in a comment | it is a `Constant` node or it is not |

That third row is Exercise 1, four cells from now.

### Cyclomatic complexity, in one sentence

Now radon's metric is trivial to state. Cyclomatic complexity (McCabe, 1976) is:

> **Start at 1. Add 1 for every decision node in the tree.**

A decision node is anything that creates a new path: `if`, `for`, `while`, `except`, `with`, a comprehension —
plus each `and` / `or`, because they short-circuit, so each one is a hidden branch. (That `BoolOp(op=And)`
above is worth +1 all by itself.)

Why it matters: **CC is a lower bound on the number of test cases needed for full path coverage.** A function
with CC 7 needs at least 7 tests to exercise every route. High CC predicts bugs not because complicated code
is ugly, but because it is *under-tested by default*.

Twelve lines of Python's built-in `ast` module reproduce radon well enough to see the idea:

In [ ]:
# See the tree for yourself. `ast` is in the standard library — nothing to install.
SNIPPET = """
if qty > 10 and price < 5:
    total = total * 0.9
"""

tree = ast.parse(SNIPPET)
print(ast.dump(tree, indent=2)[:600], "...\n")

# Formatting is invisible to the parser — these two are byte-identical as trees:
a = ast.dump(ast.parse("if x>10:\n    y=1"))
b = ast.dump(ast.parse("if (\n    x > 10\n):\n        y = 1  # a comment"))
print("same tree despite different formatting:", a == b)

# ...and the word "if" inside a string is NOT a branch, which no regex gets right for free:
print("branches found in a string literal:",
      sum(isinstance(n, ast.If) for n in ast.walk(ast.parse('s = "if this then that"'))))

In [ ]:
DECISION_NODES = (ast.If, ast.For, ast.While, ast.ExceptHandler,
                  ast.With, ast.Assert, ast.comprehension)

def my_complexity(fn_node) -> int:
    """McCabe complexity, from scratch. Start at 1, add 1 per decision point."""
    score = 1
    for node in ast.walk(fn_node):
        if isinstance(node, DECISION_NODES):
            score += 1
        elif isinstance(node, ast.BoolOp):          # `a and b and c` = 2 extra branches
            score += len(node.values) - 1
    return score

tree = ast.parse(open("inventory.py").read())
print(f"{'FUNCTION':22} {'OUR 12 LINES':>13} {'RADON':>8}")
print("-" * 46)
for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        mine = my_complexity(node)
        print(f"{node.name:22} {mine:>13} {truth.get(node.name, '—'):>8}"
              f"   {'✅' if mine == truth.get(node.name) else '≈'}")
print("\n(A '≈' is fine — radon also counts `else`/`elif` chains slightly differently.")
print(" The point is that this is ARITHMETIC. It is not something to ask a model for.)")

### Beyond Python: tree-sitter 🌳

Python's `ast` module is written in Python, for Python, and that is its limit. The moment you want to analyse
a repository that also contains TypeScript, Go and Rust, you need one parser that speaks all of them —
**[tree-sitter](https://tree-sitter.github.io/)**.

Same idea, three differences that matter in practice:

| | Python `ast` | tree-sitter |
|---|---|---|
| Languages | Python only | 100+, one grammar each, one uniform API |
| Broken code | raises `SyntaxError`, gives you nothing | parses anyway, marks the bad part `ERROR` |
| Edits | re-parse the whole file | **incremental** — re-parses only what changed, in ~1 ms |
| Fidelity | abstract: drops comments & formatting | *concrete* syntax tree: keeps every token and byte offset |
| Querying | write your own `ast.walk` loop | an S-expression query language over node patterns |

The two that change what you can *build*:

**It tolerates broken code.** Your editor must highlight and fold a file that is syntactically invalid,
because you are halfway through typing it. `ast.parse` refuses; tree-sitter returns a usable tree with an
`ERROR` node where the mess is. This is why tree-sitter is what actually runs inside editors — syntax
highlighting, code folding, and structural selection in Neovim, Zed, Atom and GitHub's own code view.

**It is incremental.** Re-parsing a 5,000-line file on every keystroke is not viable; re-parsing the ~50 bytes
that changed is. That single property is what makes tree-sitter fast enough to run *while you type*.

For agents, the relevant use is **chunking a repository for retrieval**: split at function and class
boundaries the parser found, not every 500 characters. A chunk that is a whole function is a chunk the model
can actually reason about — it is the most common quiet upgrade in a code-RAG pipeline.

```bash
pip install tree-sitter tree-sitter-python   # not needed today — this is a pointer, not a detour
```

> 🧭 **Which to reach for:** analysing Python, offline, one file at a time → the stdlib `ast` module, always.
> Polyglot repo, editor tooling, half-typed code, or splitting a codebase for retrieval → tree-sitter.

> 🧠 **The intuition to keep:** every "AI-powered code quality" feature you will ever see is some mixture of
> *a parser* (cheap, exact, boring) and *a language model* (expensive, fuzzy, insightful). Products that
> market the second while quietly relying on the first are the ones that work.

> 🧪 **Try it (2 min):** add `ast.Try` and `ast.IfExp` (ternaries) to `DECISION_NODES`. Do the numbers move
> closer to radon? Now delete `ast.BoolOp` and watch `process_order` drop. Metrics are *definitions*, not
> physical constants — which is why "our complexity went down 12%" is meaningless without naming the tool.

## 1.5 · Anatomy of an agent

Here is our minimal, framework-free agent. Production frameworks (LangGraph, AutoGen, CrewAI) add scheduling,
persistence and tracing — but **this is the essential skeleton they all share**:

| Ingredient | In our class | In production frameworks |
|---|---|---|
| **Role** | `system_prompt` | agent persona / instructions |
| **Brain** | `llm()` | model client |
| **Tools** | `{name: callable}` | tool registry / function calling |
| **Contract** | "reply ONLY with JSON" | structured output / schemas |

In [ ]:
class Agent:
    """Minimal agent: a role + a brain + tools + an output contract."""

    def __init__(self, name: str, system_prompt: str, tools: dict | None = None):
        self.name = name
        self.system_prompt = system_prompt
        self.tools = tools or {}          # {"tool_name": callable}

    def use_tools(self, *args) -> str:
        """Run every tool and concatenate the evidence."""
        report = []
        for tool_name, fn in self.tools.items():
            print(f"  🔧 {self.name} is running tool: {tool_name}")
            try:
                report.append(f"=== {tool_name} ===\n{fn(*args)}")
            except Exception as e:
                report.append(f"=== {tool_name} FAILED: {e} ===")
        return "\n\n".join(report)

    def think(self, prompt: str, **kw) -> str:
        return llm(prompt, system_prompt=self.system_prompt, **kw)

print("✅ Agent class ready")

## 1.6 · The Code Auditor

Its workflow: **(1)** run all tools → **(2)** hand the raw evidence + source to the LLM → **(3)** demand structured JSON.

Two prompt-engineering moves make the result reliable:
1. **Evidence-grounding** — the LLM *summarises tool output* instead of inventing findings from memory.
2. **A hard output contract** — JSON mode, plus the defensive `extract_json()` parser from §0.5, so the next
   agent can consume the result mechanically.

In [ ]:
AUDITOR_PROMPT = """You are a meticulous senior code reviewer.
You receive: (a) a Python source file, (b) evidence from static-analysis tools.
Identify the most important code smells / anti-patterns.

Reply with a JSON object of the form {"findings": [...]}, where each element is:
{"smell": "<short name>", "location": "<function/line>",
 "severity": "high|medium|low", "why": "<one sentence>",
 "fix": "<one-sentence refactoring suggestion>"}
List at most 6 findings, most severe first."""


# extract_json() — the defensive parser — was defined back in §0.5.
# JSON mode makes malformed output rare, not impossible: re-read that function now,
# because it is the only thing between a chatty model and a crashed pipeline.

auditor = Agent(
    name="Code Auditor",
    system_prompt=AUDITOR_PROMPT,
    tools={"radon": run_radon, "pylint": run_pylint},
)


def audit(path: str) -> list[dict]:
    evidence = auditor.use_tools(path)
    source = open(path).read()
    raw = auditor.think(
        f"SOURCE FILE ({path}):\n```python\n{source}\n```\n\n"
        f"TOOL EVIDENCE:\n{evidence}\n\nProduce the JSON findings now.",
        max_new_tokens=900, temperature=0.1, json_mode=True,
    )
    return extract_json(raw)


findings = audit("inventory.py")
findings

In [ ]:
# Pretty-print the audit like a review dashboard
SEV = {"high": "🔴", "medium": "🟠", "low": "🟡"}
print(f"{'':2} {'SMELL':28} {'WHERE':22} WHY")
print("-" * 95)
for f in findings:
    print(f"{SEV.get(f.get('severity','low'),'⚪')} {f.get('smell','?'):28.28} "
          f"{str(f.get('location','?')):22.22} {f.get('why','')[:40]}")
    print(f"{'':2} {'↳ fix:':28} {f.get('fix','')[:60]}")
cost_report()

🎉 **You built your first agent.** Deterministic tools did the measuring, the LLM did the interpreting, and a
JSON contract made the result machine-readable — ready to hand to the *next* agent in Part 2.

## 1.7 · Same skeleton, different eyes: the **ML Auditor** 🤖

Change the tools and the prompt; keep the architecture. That reuse *is* the point of the `Agent` abstraction.

In [ ]:
ML_AUDITOR_PROMPT = """You are a senior ML engineer reviewing training code.
You receive: (a) an ML training script, (b) evidence from MLScent, a static
analyser with 76 ML-specific detectors.
Prioritise: (1) silent correctness bugs, (2) reproducibility,
(3) training hygiene (early stopping, checkpoints, eval mode), (4) style.

Reply with a JSON object {"findings": [...]}, each element:
{"smell": "<short name>", "impact": "correctness|reproducibility|hygiene|style",
 "severity": "high|medium|low", "why": "<one sentence>",
 "fix": "<one-sentence fix>"}
List at most 6 findings, most severe first."""

ml_auditor = Agent(
    name="ML Auditor",
    system_prompt=ML_AUDITOR_PROMPT,
    tools={"mlscent": run_mlscent},
)


def ml_audit(project_dir: str, main_file: str) -> list[dict]:
    evidence = ml_auditor.use_tools(project_dir)
    source = open(main_file).read()
    raw = ml_auditor.think(
        f"TRAINING SCRIPT ({main_file}):\n```python\n{source}\n```\n\n"
        f"MLSCENT EVIDENCE:\n{evidence}\n\nProduce the JSON findings now.",
        max_new_tokens=900, temperature=0.1, json_mode=True,
    )
    return extract_json(raw)


ml_findings = ml_audit("ml_project", "ml_project/train_model.py")

IMPACT = {"correctness": "💥", "reproducibility": "🎲", "hygiene": "🧼", "style": "✏️"}
for f in ml_findings:
    print(f"{IMPACT.get(f.get('impact','style'),'•')} [{f.get('severity','?'):6}] "
          f"{f.get('smell','?')}: {f.get('why','')}")
    print(f"      ↳ {f.get('fix','')}")

> 💬 **Discuss (1 min):** the `== np.nan` bug means the data-cleaning branch *never runs* — yet the script trains
> and prints an accuracy. Would a code review catch it? Would your CI? What does that say about where ML
> technical debt hides?

## 1.8 · ✍️ Exercise 1 (10 min) — give the auditor a new sense

The auditor is blind to **magic numbers** as such: pylint does not flag them and radon only counts branches.
Write the missing tool, register it, and re-run the audit.

In [ ]:
import ast

def find_magic_numbers(path: str) -> str:
    """Report numeric literals that are not 0, 1, or -1 (with line numbers)."""
    tree = ast.parse(open(path).read())
    hits = []
    for node in ast.walk(tree):
        # TODO 1: check `isinstance(node, ast.Constant)` and that node.value
        #         is an int/float not in {0, 1, -1}
        # TODO 2: append f"L{node.lineno}: magic number {node.value}" to hits
        pass
    return "\n".join(hits) if hits else "no magic numbers found"

# TODO 3: register the tool and re-audit
# auditor.tools["magic_numbers"] = find_magic_numbers
# findings = audit("inventory.py")

print(find_magic_numbers("inventory.py"))

<details><summary>💡 Click for the solution</summary>

```python
def find_magic_numbers(path: str) -> str:
    tree = ast.parse(open(path).read())
    hits = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
                and node.value not in (0, 1, -1) and not isinstance(node.value, bool):
            hits.append(f"L{node.lineno}: magic number {node.value}")
    return "\n".join(hits) if hits else "no magic numbers found"

auditor.tools["magic_numbers"] = find_magic_numbers
findings = audit("inventory.py")
```
</details>

> 🧠 **The lesson behind the exercise:** you just extended an agent's *perception* without touching its brain,
> its prompt or its contract. In agentic systems, capability usually grows through **tools**, not through
> bigger models.

In [ ]:
# Save the findings — Part 2 picks them up from here
with open("findings.json", "w") as fh:
    json.dump(findings, fh, indent=2)
print("findings.json saved →", [f["smell"] for f in findings])
cost_report()
pit_stop("Part 1")

# ✅ Self-check — if this passes, you are ready for Part 2 no matter what you skipped.
assert findings and "smell" in findings[0], "re-run the audit cell in §1.6"
print("✅ Part 1 complete: an agent that measures with tools and interprets with an LLM.")

---
# Part 2 · 🤝 The Refactoring Team

**⏱ ~60 min**

In Part 1 one agent *found* problems. Now we build a **cooperating team** that *fixes* them — and, crucially,
**proves** the fix is safe before anyone accepts it.

```
        ┌──────────┐   findings   ┌────────────┐  candidate  ┌────────┐
   ───▶ │ Auditor  │ ───────────▶ │ Refactorer │ ──────────▶ │   QA   │ ──▶ ✅ / ❌
        └──────────┘              └────────────┘             └────────┘
                                        ▲                         │
                                        └──── failure notes ──────┘
```

## 2.1 · How do agents talk? The shared-state ("blackboard") pattern

| Pattern | How it works | Used by |
|---|---|---|
| **Message passing** | agents send addressed messages to each other | AutoGen-style conversations |
| **Shared state (blackboard)** | one state object every agent reads and writes | LangGraph, CrewAI, our team |

Shared state wins for *pipelines* because it is inspectable, serialisable and **replayable** — you can always
answer "why did the system do that?", which is the hardest question in agent debugging.

In [ ]:
from dataclasses import dataclass, field
import datetime

@dataclass
class WorkflowState:
    """The blackboard: one source of truth all agents read/write."""
    source_path: str
    original_code: str = ""
    candidate_code: str = ""      # the refactorer's latest proposal
    findings: list = field(default_factory=list)
    verdict: dict = field(default_factory=dict)
    accepted: bool = False
    iteration: int = 0
    history: list = field(default_factory=list)   # the audit trail

    def record(self, agent: str, event: str, detail: str = ""):
        stamp = datetime.datetime.now().strftime("%H:%M:%S")
        self.history.append(f"[{stamp}] {agent:12} | {event:22} | {detail}")
        print(self.history[-1])

state = WorkflowState(source_path="inventory.py")
state.original_code = open(state.source_path).read()
state.record("system", "state initialised", f"{len(state.original_code)} chars of smelly code")

## 2.2 · Agent 2: the **Refactoring Agent** 🔧

Its contract: *given the source + the auditor's findings, produce a complete rewritten module that fixes the
smells **without changing behaviour***.

Three guardrails make this reliable — read the prompt carefully, every line earns its place:
1. **Explicit API freeze** — name the class and the methods that must survive.
2. **Behaviour freeze** — same returns for the same inputs, spelled out.
3. **A single output format** — one fenced code block, so parsing is mechanical.

In [ ]:
REFACTORER_PROMPT = """You are an expert Python refactoring engineer.
Rewrite the ENTIRE module to fix the reported code smells.

HARD RULES — violating any of these makes your output worthless:
1. Public API must not change: class InventoryManager with methods
   __init__(items=None), add_item, process_order, refund_order, get_stock.
2. Behaviour must be IDENTICAL: same return values for the same inputs,
   including all totals, discounts, tax and status strings.
3. Use ONLY the Python standard library. Do NOT invent helper packages.
4. Replace magic numbers with named module-level constants.
5. Extract duplicated pricing logic into one private helper method.
6. Fix the mutable default argument. Remove dead code.

Output ONLY one fenced python code block containing the complete module.
No explanations before or after."""


def extract_code_block(text: str) -> str:
    """Pull the (last) fenced python block out of an LLM reply."""
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if not blocks:
        raise ValueError("Refactorer produced no code block")
    return blocks[-1].strip() + "\n"


def refactor(state: WorkflowState) -> WorkflowState:
    state.record("refactorer", "thinking", f"{len(state.findings)} findings to fix")
    findings_txt = json.dumps(state.findings, indent=1)
    reply = llm(
        f"MODULE TO REFACTOR:\n```python\n{state.original_code}\n```\n\n"
        f"AUDITOR FINDINGS:\n{findings_txt}\n\nRewrite the full module now.",
        system_prompt=REFACTORER_PROMPT, max_new_tokens=2000, temperature=0.1,
    )
    state.candidate_code = extract_code_block(reply)
    state.record("refactorer", "candidate produced", f"{len(state.candidate_code)} chars")
    return state

print("✅ refactorer ready")

## 2.3 · Agent 3: the **QA Verification Agent** 🛡️

This agent is the *only* reason we can trust anything the refactorer produces. It runs three gates,
**cheapest first** — fail fast, spend compute only on survivors:

| Gate | Cost | Catches |
|---|---|---|
| 1 · `ast.parse` | microseconds | broken syntax, truncated output |
| 2 · `pytest` in a sandbox | ~1 second | **behaviour changes** — the dangerous ones |
| 3 · complexity delta | ~1 second | "refactorings" that made things worse |

> 🔑 **Note what is missing from this agent: an LLM.** The verifier is pure, boring, deterministic code.
> Never let the thing that might hallucinate also be the thing that decides whether it hallucinated.

In [ ]:
import shutil, tempfile, os, sys

def avg_complexity(path: str) -> float:
    """Average cyclomatic complexity of a file (via radon's JSON output)."""
    raw = subprocess.run(["radon", "cc", "-j", path], capture_output=True, text=True).stdout
    data = json.loads(raw or "{}")
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return sum(scores) / len(scores) if scores else 0.0


def qa_verify(state: WorkflowState) -> WorkflowState:
    verdict = {"syntax": False, "tests": False, "improved": False, "notes": []}

    # ---- Gate 1: does it even parse? ----
    try:
        ast.parse(state.candidate_code)
        verdict["syntax"] = True
    except SyntaxError as e:
        verdict["notes"].append(f"SyntaxError: {e}")
        state.verdict = verdict
        state.record("qa", "❌ REJECTED at gate 1", str(e)[:60])
        return state

    # ---- Gate 2: behaviour preserved? (run tests in a sandbox) ----
    with tempfile.TemporaryDirectory() as sandbox:
        open(os.path.join(sandbox, "inventory.py"), "w").write(state.candidate_code)
        shutil.copy("test_inventory.py", sandbox)
        r = subprocess.run([sys.executable, "-m", "pytest", "test_inventory.py", "-q"],
                           cwd=sandbox, capture_output=True, text=True, timeout=180)
        verdict["tests"] = (r.returncode == 0)
        if not verdict["tests"]:
            verdict["notes"].append("pytest output:\n" + r.stdout[-800:])
            state.verdict = verdict
            state.record("qa", "❌ REJECTED at gate 2", "tests failed")
            return state

        # ---- Gate 3: is it actually better? ----
        cc_before = avg_complexity(state.source_path)
        cc_after = avg_complexity(os.path.join(sandbox, "inventory.py"))
        verdict["cc_before"], verdict["cc_after"] = round(cc_before, 2), round(cc_after, 2)
        verdict["improved"] = cc_after <= cc_before
        if not verdict["improved"]:
            verdict["notes"].append(f"complexity got WORSE: {cc_before} → {cc_after}")

    state.verdict = verdict
    ok = verdict["syntax"] and verdict["tests"] and verdict["improved"]
    state.record("qa", "✅ ACCEPTED" if ok else "❌ REJECTED at gate 3",
                 f"CC {verdict.get('cc_before','?')} → {verdict.get('cc_after','?')}")
    return state

print("✅ QA gates ready — no LLM inside")

## 2.4 · Under the hood: what the sandbox actually protects you from

Gate 2 does something easy to skim past: it copies the candidate into a **fresh temporary directory** and runs
pytest *there*, rather than overwriting `inventory.py` and testing in place. Three reasons, all learned the
expensive way by somebody:

**1 · An unverified patch must never touch the real file.** If you overwrite first and test second, a failing
patch has already destroyed the working code. Test-then-commit; never commit-then-test.

**2 · Python caches modules.** `import inventory` puts the module object in `sys.modules` and its compiled form
in `__pycache__`. Overwrite the file in a running kernel and a re-import may hand you the **old** code — so your
tests pass against code you already replaced. A separate directory and a separate process sidestep both caches.

**3 · Isolation makes the result meaningful.** A subprocess with its own cwd cannot be polluted by state that
earlier cells left lying around. If the tests pass there, they pass for reasons you can name.

Let's prove #2, because it is the one that silently ruins home-made pipelines:

In [ ]:
# Write a module, import it, then change it on disk — and watch Python lie to you.
pathlib.Path("cache_demo.py").write_text("VALUE = 'original'\n")
import cache_demo
print("1. after first import        :", cache_demo.VALUE)

pathlib.Path("cache_demo.py").write_text("VALUE = 'PATCHED'\n")
import cache_demo                                    # looks like a re-import; is a no-op
print("2. after editing + re-import :", cache_demo.VALUE, "  ← still stale!")

r = subprocess.run([sys.executable, "-c",
                    "import cache_demo; print(cache_demo.VALUE)"],
                   capture_output=True, text=True)
print("3. what a FRESH process sees :", r.stdout.strip(), "  ← the truth")
print("\nGate 2 always asks a fresh process. That is why its verdict can be trusted.")
os.remove("cache_demo.py")

> 💬 **Discuss (90 s):** an agent framework offers you an "LLM judge" that reads the diff and rates it 1–10
> instead of running tests. It is faster and needs no test suite. Name two things it structurally cannot catch.
> *(Anything requiring execution: a changed constant that only shows up in arithmetic; an import that fails at
> runtime; a subtle off-by-one. And it shares the failure modes of the thing it is judging.)*

## 2.5 · The **Orchestrator**: closing the loop 🔁

The orchestrator adds the two properties every agentic system needs:

- **A retry loop** — if QA rejects, the refactorer gets another shot **with the failure notes fed back**.
  That feedback is what makes iteration 2 smarter than iteration 1.
- **A hard stop** — `max_iterations`. Without it, a confused agent burns your budget forever.

In [ ]:
def run_workflow(source_path: str, max_iterations: int = 3) -> WorkflowState:
    state = WorkflowState(source_path=source_path)
    state.original_code = open(source_path).read()

    state.record("orchestrator", "▶ audit phase")
    state.findings = audit(source_path)
    state.record("auditor", "findings", ", ".join(f["smell"] for f in state.findings)[:80])

    feedback = ""
    for state.iteration in range(1, max_iterations + 1):
        state.record("orchestrator", f"▶ iteration {state.iteration}/{max_iterations}")

        if feedback:   # feed QA failure notes back to the refactorer
            state.findings = state.findings + [
                {"smell": "PREVIOUS ATTEMPT FAILED QA", "why": feedback[:400],
                 "fix": "produce a corrected complete module"}]

        state = refactor(state)
        state = qa_verify(state)

        v = state.verdict
        if v.get("syntax") and v.get("tests") and v.get("improved"):
            state.accepted = True
            break
        feedback = " | ".join(v.get("notes", []))[:400]

    state.record("orchestrator",
                 "🏁 DONE — patch " + ("ACCEPTED" if state.accepted else "NOT accepted"),
                 f"after {state.iteration} iteration(s)")
    return state


state = run_workflow("inventory.py", max_iterations=3)
cost_report()

### 🔍 Inspect the result

> ⚠️ **It is fine (and instructive) if an iteration was rejected.** Models *do* break behaviour — and your QA
> gate just caught it in front of your eyes. That catch **is** the lesson: the gate, not the model, is what
> makes the system trustworthy.

In [ ]:
import difflib

if state.accepted:
    diff = difflib.unified_diff(
        state.original_code.splitlines(keepends=True),
        state.candidate_code.splitlines(keepends=True),
        fromfile="inventory.py (before)", tofile="inventory.py (after)")
    print("".join(diff))
else:
    print("No accepted patch — inspect state.verdict['notes'] to see why:")
    print("\n".join(state.verdict.get("notes", []))[:1200])

In [ ]:
# Before / after scoreboard
mi = lambda p: subprocess.run(["radon", "mi", "-s", p], capture_output=True, text=True).stdout.strip()
print("METRIC                BEFORE                       AFTER")
print("-" * 70)
if state.accepted:
    open("inventory_refactored.py", "w").write(state.candidate_code)
    print(f"avg complexity        {state.verdict['cc_before']:<28} {state.verdict['cc_after']}")
    print(f"maintainability       {mi('inventory.py'):<28.28} {mi('inventory_refactored.py'):<.28}")
    print(f"tests                 7 passed                     7 passed  ✅ behaviour preserved")
else:
    print("(accept a patch first)")

In [ ]:
# The audit trail: every decision, replayable. THIS is how you debug agents.
print("\n".join(state.history))
pit_stop("Part 2")

## 2.6 · Why the retry loop converges

The orchestrator does one thing that looks trivial and carries all the weight:

```python
if feedback:
    state.findings = state.findings + [
        {"smell": "PREVIOUS ATTEMPT FAILED QA", "why": feedback[:400], ...}]
```

It appends **the actual failure text** — the pytest output, the SyntaxError, the complexity numbers — to the
next prompt. Without it, iteration 2 is just iteration 1 rolled again at the same temperature: same input,
same distribution, same class of mistake. *That* is the difference between a retry loop and a feedback loop.

Think of it as the difference between:

| | Retry (no feedback) | Feedback loop (ours) |
|---|---|---|
| Prompt on attempt 2 | identical to attempt 1 | attempt 1 **+ why it failed** |
| Expected outcome | same error, resampled | error is *addressed* |
| Converges? | only by luck | usually within 2–3 tries |
| Cost of failure | pure waste | information |

Crucially, the feedback is **generated by a deterministic tool**, not by another model. pytest's output is
ground truth about the candidate. We are not asking one LLM to critique another — we are handing the LLM a
measurement it cannot argue with.

> 🧠 **The general principle:** *an agent is only as good as the error signal you feed back into it.* Most
> disappointing agent systems have a perfectly good loop wrapped around a signal that says nothing more
> specific than "that didn't work".

Look at what the refactorer would actually receive on a second attempt:

In [ ]:
# Construct a deliberately failing candidate, gate it, and print the feedback the loop would send back.
demo = WorkflowState(source_path="inventory.py")
demo.original_code = open("inventory.py").read()
demo.candidate_code = demo.original_code.replace("total * 0.25", "total * 0.30")  # wrong VAT
demo = qa_verify(demo)

feedback = " | ".join(demo.verdict.get("notes", []))[:400]
print("─" * 70)
print("WHAT ITERATION 2's PROMPT WOULD GAIN:\n")
print(feedback[:600])
print("─" * 70)
print("\nNotice: exact assertion, expected vs actual value, test name. The model does not")
print("have to GUESS what went wrong — and that is why iteration 2 usually fixes it.")

## 2.7 · 🔥 The Failure-Pattern Lab

Time to break things on purpose. Agentic systems fail in *recurring, nameable* ways. Knowing the names is half the defence.

| # | Failure pattern | What it looks like | Mitigation (which we built) |
|---|---|---|---|
| 1 | **Plausible-but-wrong code** | patch looks perfect, silently changes a constant | gate 2 — behaviour tests |
| 2 | **Contract drift** | model returns prose instead of a code block | strict parser + retry |
| 3 | **Reward hacking** | "simplifies" by deleting features | gate 2 + gate 3 together |
| 4 | **Runaway loops** | agent retries forever | `max_iterations` |
| 5 | **Unverifiable claims** | "I refactored it safely!" | no LLM inside the verifier |

### The sabotage demo

Below we hand QA a patch where a single VAT constant was changed from `0.25` to `0.20`. It parses. It looks
clean. It is a €-losing production bug.

> 🎯 **Predict first:** set `MY_GUESS_CAUGHT` in the next cell. Does gate 2 catch it — and if so, do you
> expect the failure notes to name the *exact* wrong number, or just say "a test failed"? The difference
> decides whether iteration 2 can fix it.

In [ ]:
MY_GUESS_CAUGHT = True   # 🎯 EDIT ME: will the QA gate reject this sabotage? True / False

sabotage = state.original_code.replace("total * 0.25", "total * 0.20")  # 🕵️ subtle!

evil_state = WorkflowState(source_path="inventory.py")
evil_state.original_code = state.original_code
evil_state.candidate_code = sabotage
evil_state = qa_verify(evil_state)

print("\nQA verdict on the sabotaged patch:")
print(json.dumps({k: v for k, v in evil_state.verdict.items() if k != "notes"}, indent=2))
print("\nFailure notes:\n", "\n".join(evil_state.verdict["notes"])[:600])

quiz("Does the gate catch a one-constant sabotage?",
     MY_GUESS_CAUGHT, not evil_state.verdict["tests"],
     "The gate does not 'review' the patch — it RUNS it. That is why it catches what looks fine and "
     "would miss what merely looks ugly. Cheap, deterministic, and completely immune to a persuasive diff.")

☝️ The tests caught in ~1 second what code review would probably miss. Now flip it around:

> 💬 **Discuss (2 min):** gate 2 is only as strong as the test suite. What sabotage would slip through *our*
> seven tests? *(Hint: is `add_item`'s discount parameter tested at all? What about float edge cases?)*
> This is exactly why **test debt is the most expensive debt** in an agentic pipeline — it silently lowers the
> ceiling on everything an agent is allowed to do.

## 2.8 · ✍️ Exercise 2 (15 min) — pick one

**A · The Documenter agent.** Add a 4th agent that adds Google-style docstrings to the accepted module *without
changing a single identifier or expression* — then push its output through `qa_verify` like everyone else.

**B · A 4th gate.** Extend `qa_verify` so the maintainability index must also improve, not just complexity.
Does the pipeline still accept anything? What does that tell you about setting quality bars?

**C · Break the refactorer.** Delete one HARD RULE from `REFACTORER_PROMPT` and re-run. Which rule was
load-bearing?

In [ ]:
# 🖊️ Your Exercise 2 workspace

DOCSTRING_PROMPT = """You are a documentation engineer. Add concise Google-style
docstrings to every class and method. Change NOTHING else — not one identifier,
not one expression. Output ONLY one fenced python code block."""

def document(state):
    # TODO: call llm() with DOCSTRING_PROMPT + state.candidate_code,
    #       extract_code_block(), put the result back into state.candidate_code,
    #       then re-run qa_verify(state) and keep it ONLY if accepted.
    pass

<details><summary>💡 Solution sketch (option A)</summary>

```python
def document(state):
    reply = llm(f"```python\n{state.candidate_code}\n```",
                system_prompt=DOCSTRING_PROMPT, max_new_tokens=2000, temperature=0.1)
    proposal = extract_code_block(reply)
    trial = WorkflowState(source_path=state.source_path)
    trial.original_code = state.original_code
    trial.candidate_code = proposal
    trial = qa_verify(trial)
    if trial.verdict.get("tests"):          # docstrings must not change behaviour
        state.candidate_code = proposal
        state.record("documenter", "✅ docstrings accepted")
    else:
        state.record("documenter", "❌ rejected — behaviour changed")
    return state

state = document(state)
```
Note the pattern: **a new agent does not get a new trust level.** It goes through the same gate.
</details>

---
# Part 3 · 💳 Technical Debt: Classify, Triage, Resolve

**⏱ ~40 min**

Parts 1–2 operated on **code**. But most technical debt is first reported in **words** — issue trackers, PR
comments, TODO notes. In this part the team learns to read the backlog, price it, and act on it.

**Interest vs principal** — the metaphor that makes debt a management conversation instead of an engineering complaint:
- **principal** = what the proper fix costs, once.
- **interest** = what the shortcut costs you, *every sprint until then*.

## 3.1 · A slice of a real-world issue tracker

Ten issues of the kind **BEACon-TD** (JSS 2025) was trained on. Each hides a debt type — sometimes several.
We keep hand labels so we can *score* our classifier instead of admiring it.

In [ ]:
ISSUES = [
 {"id": 101, "text": "The OrderService class is 3000 lines and does payment, shipping AND emails. Every change breaks something unrelated. We need to split it before adding new payment providers.", "gold": "design"},
 {"id": 102, "text": "There are zero tests for the refund flow. We only find regressions when customers complain. Adding tests keeps getting postponed for feature work.", "gold": "test"},
 {"id": 103, "text": "The README still describes the v1 API. New joiners lose days because the setup guide is wrong and the architecture diagram shows services we deleted last year.", "gold": "documentation"},
 {"id": 104, "text": "We're pinned to Django 2.2 which reached end-of-life. Security patches no longer land and two dependencies refuse to install alongside it.", "gold": "dependency"},
 {"id": 105, "text": "TODO left from the March crunch: error handling in the CSV importer just swallows exceptions with a bare except and returns None. Works until it doesn't.", "gold": "defect"},
 {"id": 106, "text": "The nightly ETL takes 6 hours because it re-reads the entire table every run. A watermark column was proposed in 2023 but never implemented.", "gold": "design"},
 {"id": 107, "text": "Deployment is a 14-step manual runbook involving three people and an SSH session. One typo in step 9 took prod down last month. We need CI/CD.", "gold": "build"},
 {"id": 108, "text": "Variable names in the pricing module are a, b, tmp2 and data_final_v3. Code review of any pricing change takes twice as long as it should.", "gold": "code"},
 {"id": 109, "text": "Our fork of the auth library diverged 200 commits from upstream. Merging upstream security fixes now takes a full sprint each time.", "gold": "dependency"},
 {"id": 110, "text": "The ML model in production was trained on 2022 data and nobody saved the training script or the random seed. Retraining reproducibly is currently impossible.", "gold": "code"},
]
print(f"{len(ISSUES)} issues loaded")

## 3.2 · Agent 4: the **TD Classifier**

BEACon-TD's production answer is a *fine-tuned transformer* — small, fast, cheap, consistent. Today we
approximate the task with a **constrained-label prompt** (zero-shot classification).

Note the constraint pattern: give the model a *closed* label set, demand one word, and **snap the answer back
into the set** in code. Never trust free text where an enum belongs.

> 🎯 **Predict first:** ten issues, eight labels, no training examples. Set `MY_GUESS_ACCURACY` to the
> percentage you expect the zero-shot classifier to get right. Random guessing would score ~12%.

In [ ]:
MY_GUESS_ACCURACY = 70   # 🎯 EDIT ME (0–100): what % of the 10 issues will it label correctly?

LABELS = ["design", "code", "test", "documentation",
          "dependency", "build", "defect", "requirement"]

CLASSIFIER_PROMPT = f"""You classify technical-debt reports from issue trackers.
Allowed labels (choose EXACTLY one): {", ".join(LABELS)}.

Definitions:
- design: architectural problems, god classes, wrong abstractions, inefficient designs
- code: poor readability/naming, code smells, missing reproducibility of code artifacts
- test: missing/weak/flaky tests, low coverage
- documentation: missing or outdated docs/diagrams/guides
- dependency: outdated/EOL libraries, diverged forks, version conflicts
- build: manual/fragile build, deployment or CI/CD problems
- defect: known bugs or error-handling gaps deliberately left in the code
- requirement: implementation diverged from what was actually required

Reply with ONLY the label word. Nothing else."""


def classify_issue(text: str) -> str:
    reply = llm(f"ISSUE:\n{text}\n\nLabel:", system_prompt=CLASSIFIER_PROMPT,
                max_new_tokens=8, temperature=0.0)
    reply = reply.lower().strip().split()[0].strip(".,:")
    return reply if reply in LABELS else "code"   # snap to label set


import pandas as pd
rows = []
for issue in ISSUES:
    pred = classify_issue(issue["text"])
    rows.append({"id": issue["id"], "gold": issue["gold"], "predicted": pred,
                 "✓": "✅" if pred == issue["gold"] else "❌",
                 "issue": issue["text"][:70] + "…"})
df = pd.DataFrame(rows)
accuracy = (df["gold"] == df["predicted"]).mean()
print(f"Zero-shot accuracy: {accuracy:.0%}\n")

quiz(f"You predicted {MY_GUESS_ACCURACY}% accuracy. Actual: {accuracy:.0%}.",
     MY_GUESS_ACCURACY, lambda g: abs(g - accuracy * 100) <= 15,
     "A closed label set plus explicit definitions is a genuinely strong zero-shot recipe — the constraint "
     "is doing as much work as the model is. " +
     ("A clean sweep on ten issues is NOT proof it generalises: ten hand-picked examples is a demo, "
      "not an evaluation. Change one label definition above and watch the score move."
      if accuracy == 1.0 else
      "Now look at WHICH ones it missed — ambiguity in the labels, not weakness in the model, "
      "explains most of them."))

df

### 🤔 Reading the score

**If you have ❌ rows:** look at them before looking at the number. Most misclassifications here are
*genuinely ambiguous* — issue 110 (unreproducible ML training) is arguably `code`, `design`, **and** a process
problem. Real issues carry several debt types at once, which is why BEACon-TD treats this as **multi-label**
classification over 13 types rather than forcing one winner.

**If you scored 100%:** resist the celebration. Ten hand-picked issues with clean, distinct wording is a
*demo*, not an evaluation. A real backlog is full of two-line tickets that say "fix the thing in checkout",
and your accuracy there is unknown until you measure it on *your* data. The lesson is the recipe, not the
score: **closed label set + explicit definitions + snap-back in code**.

> 🧪 **Try it (2 min):** delete the `- design:` definition line from `CLASSIFIER_PROMPT` and re-run. Watch how
> much of the accuracy was coming from the *prompt's* definitions rather than from the model's world
> knowledge. That is the cheapest lesson in prompt engineering you will get today.

> 💬 **Discuss (2 min):** a fine-tuned BEACon-TD model costs ~1000× less per issue, runs on-prem, and —
> crucially — gives the *same* answer tomorrow. When is prompting the right call, and when do you fine-tune?

## 3.3 · Under the hood: why a closed label set works

The classifier prompt does something worth naming, because it generalises far beyond debt classification:

```python
Allowed labels (choose EXACTLY one): design, code, test, ...
Reply with ONLY the label word.
```
```python
return reply if reply in LABELS else "code"     # snap to the label set
```

That is **three independent layers of constraint**, and you need all three:

| Layer | Mechanism | Catches |
|---|---|---|
| 1 · Enumerate the labels | the model has seen the vocabulary it must use | "architectural-debt", "tech debt (design)" |
| 2 · `max_new_tokens=8` | it *cannot* produce a paragraph | "This issue primarily concerns…" |
| 3 · `if reply in LABELS` | code, not hope | anything the first two missed |

Layer 3 is the one people skip, and it is the only one that is a *guarantee*. Layers 1 and 2 make the good
outcome likely; layer 3 makes the bad outcome **impossible**. Never ship a pipeline whose correctness depends
only on the model complying.

> ⚠️ **But look closely at our layer 3:** `else "code"`. A parse failure is silently relabelled as `code` — so
> a broken classifier reports as a *confident* one. That is a real design smell in our own notebook. In
> production you would return `None` and count it, so you can see your failure rate instead of laundering it
> into your results.

> 🧪 **Try it (2 min):** change the fallback to `else "PARSE_FAILED"` and re-run §3.2. How many are there?
> Now delete the label list from the prompt (keep layer 3) and re-run. Watch the failure count climb — you
> have just measured what layer 1 was worth.

### Why production uses a fine-tuned model instead

BEACon-TD does not prompt a general model; it fine-tunes a small transformer on labelled issues. The trade:

| | Prompted general model (today) | Fine-tuned small model (BEACon-TD) |
|---|---|---|
| Setup cost | minutes | needs a labelled dataset |
| Per-issue cost | ~$0.0002 | ~1000× less; runs on CPU |
| Consistency | drifts with prompt wording and model updates | fixed weights → same answer next year |
| Privacy | issues leave your network | runs on-prem |
| New label | edit a string | retrain |

Prompting wins for exploration and for label sets that change weekly. Fine-tuning wins the moment the task is
stable, high-volume, or sensitive — which describes most production classification.

## 3.4 · Agent 5: the **Triage Agent** — pricing the debt

In [ ]:
TRIAGE_PROMPT = """You are an engineering manager triaging technical debt.
For the issue, estimate:
- interest: recurring pain per sprint if NOT fixed, integer 1 (minor) to 5 (severe)
- principal: effort of the proper fix, integer 1 (hours) to 5 (multi-sprint)
- rationale: one short sentence

Reply ONLY with a JSON object: {"interest": int, "principal": int, "rationale": str}"""


def triage(issue) -> dict:
    reply = llm(f"ISSUE:\n{issue['text']}", system_prompt=TRIAGE_PROMPT,
                max_new_tokens=160, temperature=0.1, json_mode=True)
    try:
        score = json.loads(reply)
    except json.JSONDecodeError:
        score = {"interest": 3, "principal": 3, "rationale": "parse failed"}
    # priority = pain per unit of effort
    score["priority"] = round(score.get("interest", 3) / max(score.get("principal", 3), 1), 2)
    return score


backlog = []
for issue in ISSUES:
    s = triage(issue)
    backlog.append({"id": issue["id"], "type": classify_issue(issue["text"]),
                    "interest": s.get("interest"), "principal": s.get("principal"),
                    "priority": s["priority"], "rationale": s.get("rationale", "")[:60]})

pd.DataFrame(sorted(backlog, key=lambda r: -r["priority"]))

> 💬 **Discuss (2 min):** the LLM just made **resource-allocation judgements**. Would you let this ranking
> drive sprint planning directly? What human checkpoint would you insert, and *why exactly there*?
> *(No consensus answer — the trade-off between automation speed and accountable judgement is the point.)*

## 3.5 · The debt quadrant: turning two numbers into a decision

`priority = interest ÷ principal` compresses the triage agent's two estimates into one ranking. That is
convenient, and it throws away the shape of the problem. The quadrant keeps it:

```
        high interest
              ▲
   FIX NOW    │   SCHEDULE
   quick wins │   big, unavoidable
  ────────────┼────────────▶  high principal
   WHENEVER   │   TOLERATE
   cheap, dull│   expensive & harmless
```

- **Fix now** (high interest, low principal) — pay these first. Highest return per hour.
- **Schedule** (high interest, high principal) — these never "fit between features". They need a plan and a
  budget, and they are what a debt *strategy* is actually for.
- **Whenever** (low interest, low principal) — good onboarding tasks.
- **Tolerate** (low interest, high principal) — the hardest quadrant, because it feels irresponsible. It is
  not. Debt you are not paying interest on is **not worth paying off**; the ugly module nobody touches can
  stay ugly.

That last one is why `priority = interest ÷ principal` alone is not enough: it would rank a *Tolerate* item
above a *Schedule* item whenever the arithmetic happened to work out. Ratios rank; quadrants decide.

In [ ]:
# Place the backlog on the quadrant (no plotting libraries — just the two numbers and a threshold)
QUAD = {(True, False): "🔥 FIX NOW ", (True, True): "🗓  SCHEDULE",
        (False, False): "🧹 WHENEVER", (False, True): "😐 TOLERATE "}

print(f"{'QUADRANT':12} {'ID':>5} {'int':>4} {'prn':>4} {'ratio':>6}  ISSUE")
print("-" * 88)
for r in sorted(backlog, key=lambda r: -r["priority"]):
    hi_interest = (r["interest"] or 3) >= 4
    hi_principal = (r["principal"] or 3) >= 4
    text = next(i["text"] for i in ISSUES if i["id"] == r["id"])
    print(f"{QUAD[(hi_interest, hi_principal)]:12} {r['id']:>5} {r['interest']:>4} "
          f"{r['principal']:>4} {r['priority']:>6}  {text[:48]}…")

print("\n💬 Find an item the RATIO ranks high but the QUADRANT says 'Tolerate'.")
print("   Would you actually schedule it? That gap is why humans still run planning.")

## 3.6 · 🏆 The capstone: the full pipeline

Everything you built, in one function: classify → triage → audit → refactor ⇄ QA → **report**.

In [ ]:
def full_pipeline(code_path: str, issues: list, max_iterations: int = 3) -> str:
    """Issues + code in → verified fix + Markdown debt report out."""
    log = lambda *a: print("  ", *a)

    print("① Classifying & triaging the issue backlog …")
    ranked = []
    for it in issues:
        s = triage(it)
        ranked.append({**it, "type": classify_issue(it["text"]), **s})
    ranked.sort(key=lambda r: -r["priority"])

    print("② Auditing the code …")
    code_findings = audit(code_path)
    log(f"{len(code_findings)} findings:", ", ".join(f.get("smell", "?") for f in code_findings)[:80])

    print("③ Refactor ⇄ QA loop …")
    st = WorkflowState(source_path=code_path)
    st.original_code = open(code_path).read()
    st.findings = code_findings
    feedback = ""
    for st.iteration in range(1, max_iterations + 1):
        if feedback:
            st.findings = code_findings + [{"smell": "PREVIOUS ATTEMPT FAILED QA",
                                            "why": feedback[:400],
                                            "fix": "produce a corrected complete module"}]
        try:
            st = refactor(st)
        except ValueError as e:
            feedback = str(e); log(f"iter {st.iteration}: ❌ {e}"); continue
        st = qa_verify(st)
        v = st.verdict
        if v.get("syntax") and v.get("tests") and v.get("improved"):
            st.accepted = True
            log(f"iter {st.iteration}: ✅ accepted")
            break
        feedback = " | ".join(v.get("notes", []))[:400]
        log(f"iter {st.iteration}: ❌ {feedback[:70]}")

    print("④ Writing TECH_DEBT_REPORT.md …")
    lines = ["# Technical Debt Report", "",
             f"*Generated by a cooperative LLM-agent pipeline · model: {MODEL_NAME}*", "",
             "## 1 · Prioritised issue backlog (top 5)", "",
             "| Rank | Issue | Type | Interest | Principal | Priority |", "|--|--|--|--|--|--|"]
    for rank, r in enumerate(ranked[:5], 1):
        lines.append(f"| {rank} | #{r['id']} {r['text'][:55]}… | {r['type']} | "
                     f"{r['interest']} | {r['principal']} | {r['priority']} |")

    lines += ["", "## 2 · Code audit findings", ""]
    for f in code_findings:
        lines.append(f"- **{f.get('smell','?')}** ({f.get('severity','?')}, {f.get('location','?')}): "
                     f"{f.get('why','')} → *{f.get('fix','')}*")

    lines += ["", "## 3 · Automated refactoring outcome", ""]
    if st.accepted:
        open("inventory_refactored.py", "w").write(st.candidate_code)
        lines += [f"- ✅ Patch **accepted**: all 7 behaviour tests pass; "
                  f"avg complexity {st.verdict['cc_before']} → {st.verdict['cc_after']}",
                  "- Verified code saved to `inventory_refactored.py`"]
    else:
        lines += [f"- ❌ No patch met the QA bar within {max_iterations} iterations "
                  f"(last reason: {feedback[:120]}). Original code kept — "
                  "**the gate held; nothing unverified shipped.**"]

    print("⑤ Scanning ML code with MLScent …")
    ml_report = run_mlscent("ml_project")
    counts = ml_report.split("Smell Counts:")[-1].strip().splitlines()[:8]
    lines += ["", "## 4 · ML-specific smells (`ml_project/` · MLScent)", ""]
    lines += [f"- {c.strip()}" for c in counts if c.strip()]
    lines += ["", "*Full MLScent report: `output/analysis_report.txt` — each finding includes a How-to-fix.*"]

    report = "\n".join(lines)
    open("TECH_DEBT_REPORT.md", "w").write(report)
    return report


report = full_pipeline("inventory.py", ISSUES, max_iterations=3)
cost_report()

In [ ]:
from IPython.display import Markdown, display
display(Markdown(report))
pit_stop("Part 3")
print("✅ Part 3 complete. Part 4 is the bonus round — everything below is take-home-safe.")

🏆 **You built an end-to-end technical-debt management pipeline** — perception (tools), reasoning (LLM),
action (refactoring), verification (gates), and reporting. The **architecture** is what made it trustworthy,
not the size of the model.

## 3.7 · ✍️ Final challenge — bring your own code

Paste any Python module of yours into the cell below, **write 2–3 behaviour tests for it**, point
`qa_verify`'s sandbox at your test file, and run `full_pipeline` on it.

- No tests you trust? → notice how uncomfortable that feels. **That discomfort is test debt, quantified emotionally.**
- Model mangles your code? → the gate rejects it. Working as designed.
- Everything passes first try? → raise the bar in `qa_verify` (make MI improvement mandatory) and see what happens.

In [ ]:
%%writefile my_module.py
# 🖊️ Paste YOUR code here, then adapt test_inventory.py-style tests for it,
# point qa_verify's sandbox at your test file, and run:
#   report = full_pipeline("my_module.py", ISSUES)

def example(a, b):
    return a + b

---
# Part 4 · 🚀 Production Frameworks (bonus / take-home)

**⏱ ~50 min, self-paced**

Parts 1–3 built every pattern **by hand**, on purpose, so nothing is magic. Part 4 rebuilds the *same*
pipeline with the production stack:

| Section | Tool | What it buys you |
|---|---|---|
| 4.1 | **a graph engine in 15 lines** | demystify LangGraph by building it first |
| 4.2 | **LangGraph** | the loop as a typed `StateGraph` — free diagram, streaming, checkpointing |
| 4.3 | **Deep Agents** | autonomous planning (`write_todos`), file tools, real tool-calling |
| 4.4 | **the tool-calling protocol** | what "the agent used a tool" really means |
| 4.5 | **`debtbuster`** | the whole thing packaged as a `pip install`-able CLI that exits non-zero in CI |

> 🔑 **Watch what does NOT change:** the QA gates. They are byte-for-byte the same across all three versions,
> because **verification is a property of the task, not of the framework**.

## 4.1 · Under the hood: a graph executor in fifteen lines

Before importing LangGraph, build the thing it is. A state graph is not a deep idea — it is **a dict of named
functions, a rule for picking the next name, and a `while` loop**. Writing it once means you will never be
confused about what the framework is doing to your code.

In [ ]:
def tiny_graph(nodes: dict, router, state: dict, start: str, max_steps: int = 20) -> dict:
    """A complete state-graph engine. This is the whole concept."""
    current, steps = start, 0
    while current != "END" and steps < max_steps:        # ← the iteration budget, again
        state.update(nodes[current](state))              # node returns a PARTIAL state update
        current = router(current, state)                 # ← the conditional edge
        steps += 1
    return state

# The Auditor → Refactorer ⇄ QA loop, on the toy engine:
demo_nodes = {
    "audit":     lambda s: {"findings": "magic numbers; duplicated pricing", "n": s["n"]},
    "refactor":  lambda s: {"n": s["n"] + 1, "candidate": f"attempt {s['n'] + 1}"},
    "qa":        lambda s: {"ok": s["n"] >= 2},           # pretend attempt 2 passes
}
def demo_router(node, s):
    if node == "audit":    return "refactor"
    if node == "refactor": return "qa"
    return "END" if s["ok"] or s["n"] >= 3 else "refactor"

out = tiny_graph(demo_nodes, demo_router, {"n": 0, "ok": False}, start="audit")
print("final state:", out)
print("\nThat is LangGraph's core. Everything below is engineering, not concept:")
print("  typed state · streaming · checkpoint/resume · retries · tracing · the diagram")

### So what is LangGraph *for*?

If the engine is fifteen lines, the honest question is why take the dependency at all. The answer is that the
fifteen lines are the easy part; the properties around them are not:

| You will eventually need | Hand-rolled cost | LangGraph |
|---|---|---|
| a picture of the workflow, always current | you draw it, it rots | `get_graph().draw_mermaid_png()` |
| resume after a crash mid-run | design a checkpoint format | a checkpointer |
| stream partial results to a UI | restructure everything | built in |
| typed state, checked | discipline | `TypedDict` |
| a shape colleagues recognise | onboarding | it *is* the convention |

> ⚖️ **The lazy engineer's rule:** hand-roll it until you need the second property on that list, then adopt the
> framework. Adopting on day one costs you the understanding you just built in fifteen lines; adopting on day
> never costs you the next six months.

Now compare `tiny_graph` above with the real thing in §4.2 — you will recognise every part.

## 4.2 · The refactoring team, rebuilt in **LangGraph**

LangGraph models a workflow as **typed state + a graph of nodes**. Our `WorkflowState` becomes a `TypedDict`,
our three agents become nodes, and the retry loop becomes a **conditional edge**.

Note we keep calling our own `llm()` — no extra LangChain provider package needed for this section.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

AUDITOR_SYS = ("You are a strict senior code reviewer. Given a Python module, list its "
               "worst code smells as short bullet points (max 6). Be specific: name the "
               "function and the problem. No fixes yet, no prose intro.")
REFACTORER_SYS = ("You are a careful refactoring engineer. Rewrite the module fixing the "
                  "listed smells. HARD INVARIANTS: same public API and behaviour; stdlib only; "
                  "replace magic numbers with named constants; remove duplication and dead code. "
                  "Reply with ONE ```python``` block containing the FULL module, nothing else.")


def qa_gates(original_path: str, candidate: str) -> dict:
    """The Part-2 gates, in the shape LangGraph wants. Still no LLM inside."""
    try:
        ast.parse(candidate)                                        # gate 1
    except SyntaxError as e:
        return {"ok": False, "why": f"gate 1 (syntax): {e}"}
    with tempfile.TemporaryDirectory() as tmp:                      # gate 2
        pathlib.Path(tmp, "inventory.py").write_text(candidate)
        shutil.copy("test_inventory.py", tmp)
        r = subprocess.run([sys.executable, "-m", "pytest", "-q", "test_inventory.py"],
                           cwd=tmp, capture_output=True, text=True, timeout=180)
        if r.returncode != 0:
            return {"ok": False, "why": "gate 2 (behaviour): " + r.stdout[-300:]}
        cc_after = avg_complexity(str(pathlib.Path(tmp, "inventory.py")))
    cc_before = avg_complexity(original_path)                       # gate 3
    if cc_after > cc_before:
        return {"ok": False, "why": f"gate 3 (quality): CC worsened {cc_before:.2f} → {cc_after:.2f}"}
    return {"ok": True, "why": f"all gates passed · CC {cc_before:.2f} → {cc_after:.2f}"}


class TeamState(TypedDict):
    source: str
    findings: str
    candidate: str
    verdict: str
    accepted: bool
    iteration: int


def auditor_node(state: TeamState) -> dict:
    print("🕵️ auditor …")
    return {"findings": llm(state["source"], system_prompt=AUDITOR_SYS,
                            max_new_tokens=600, temperature=0.1)}


def refactorer_node(state: TeamState) -> dict:
    print(f"🔧 refactorer (iteration {state['iteration'] + 1}) …")
    fb = f"\nPREVIOUS ATTEMPT REJECTED: {state['verdict']}" if state["verdict"] else ""
    reply = llm(f"MODULE:\n```python\n{state['source']}\n```\nSMELLS:\n{state['findings']}{fb}",
                system_prompt=REFACTORER_SYS, max_new_tokens=2000, temperature=0.1)
    try:
        return {"candidate": extract_code_block(reply), "iteration": state["iteration"] + 1}
    except ValueError as e:
        return {"candidate": "", "verdict": str(e), "iteration": state["iteration"] + 1}


def qa_node(state: TeamState) -> dict:
    print("🛡️ qa gates …")
    if not state["candidate"]:
        return {"accepted": False}
    v = qa_gates("inventory.py", state["candidate"])
    print("   ", "✅" if v["ok"] else "❌", v["why"][:90])
    return {"accepted": v["ok"], "verdict": v["why"]}


def route_after_qa(state: TeamState) -> str:
    if state["accepted"]:
        return "done"
    return "give_up" if state["iteration"] >= 3 else "retry"


g = StateGraph(TeamState)
g.add_node("auditor", auditor_node)
g.add_node("refactorer", refactorer_node)
g.add_node("qa", qa_node)
g.add_edge(START, "auditor")
g.add_edge("auditor", "refactorer")
g.add_edge("refactorer", "qa")
g.add_conditional_edges("qa", route_after_qa, {"done": END, "retry": "refactorer", "give_up": END})
team = g.compile()
print("✅ graph compiled")

In [ ]:
# LangGraph draws its own architecture diagram — compare it to the ASCII sketch in Part 2!
from IPython.display import Image, display
try:
    display(Image(team.get_graph().draw_mermaid_png()))
except Exception:
    print(team.get_graph().draw_mermaid())   # fallback: raw mermaid text

In [ ]:
result = team.invoke({"source": open("inventory.py").read(),
                      "findings": "", "candidate": "", "verdict": "",
                      "accepted": False, "iteration": 0})

print("\n" + "=" * 60)
if result["accepted"]:
    open("inventory_langgraph.py", "w").write(result["candidate"])
    print(f"✅ ACCEPTED after {result['iteration']} iteration(s) — {result['verdict']}")
    print("→ saved to inventory_langgraph.py")
else:
    print(f"❌ REJECTED after {result['iteration']} iteration(s) — {result['verdict'][:200]}")
    print("(the gate held; nothing unverified shipped)")
cost_report()

> 💬 **Compare (2 min):** put Part 2's `run_workflow()` next to this graph. Same auditor, same gates, same loop.
> **What did LangGraph buy you?** The diagram for free, typed state, streaming/checkpointing when you need it,
> and a shape your colleagues already recognise. **What did it cost?** A dependency, an abstraction layer, and
> a debugging story that now runs through someone else's code.

## 4.3 · An autonomous **Deep Agent** 🤖

Everything so far had a *fixed* control flow: we decided the order. A **deep agent** decides for itself — it
plans with `write_todos`, calls tools in whatever order it judges useful, reads and writes files, and stops
when it thinks it is done.

`deepagents` talks to models through LangChain, so this one section needs the OpenAI binding package:

In [ ]:
%pip install -q langchain-openai
print("✅ deepagents can now reach OpenAI")

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend


# ⚠️ Two guards every autonomous-agent tool needs. Learned the hard way while building
#    this notebook: the first version let the agent call radon_report("/") — it walked the
#    ENTIRE filesystem and returned a 6.8 MB "tool result". Unbounded tools are a failure
#    pattern, not an edge case.
def _safe(path: str) -> str:
    """Guard 1 — keep the agent inside the workshop directory."""
    p = pathlib.Path(path)
    p = (p if p.is_absolute() else WORKDIR / p).resolve()
    return str(p) if p == WORKDIR or WORKDIR in p.parents else str(WORKDIR)


LIMIT = 3000                       # Guard 2 — every tool result is truncated


def radon_report(path: str) -> str:
    """Cyclomatic-complexity report for a Python file or directory."""
    out = subprocess.run(["radon", "cc", "-s", _safe(path)],
                         capture_output=True, text=True, timeout=120).stdout
    return out[:LIMIT] or "n/a"


def pyexamine_report(path: str) -> str:
    """PyExamine (MSR 2025): 49-metric code-smell report for a directory."""
    subprocess.run(["analyze_code_quality", _safe(path), "--type", "code", "--output", "pyx"],
                   capture_output=True, text=True, timeout=600)
    p = pathlib.Path("pyx.txt")
    return p.read_text()[:LIMIT] if p.exists() else "no report produced"


def mlscent_report(path: str) -> str:
    """MLScent (CAIN 2025): 76 ML-specific anti-pattern detectors for a directory."""
    subprocess.run(["ml_smell_detector", "analyze", _safe(path)],
                   capture_output=True, text=True, timeout=600)
    p = pathlib.Path("output/analysis_report.txt")
    return p.read_text()[:LIMIT] if p.exists() else "no report produced"


deep_auditor = create_deep_agent(
    model=f"openai:{MODEL_NAME}",
    tools=[radon_report, pyexamine_report, mlscent_report],
    backend=FilesystemBackend(root_dir=str(WORKDIR)),   # real files, sandboxed to our workdir
    system_prompt=(
        "You are an autonomous code-quality auditor. Plan your work with write_todos first. "
        "Your filesystem root IS the working directory: the files are 'inventory.py' and "
        "'ml_project/'. Do not go looking for them elsewhere — no /workspace, /root, /app. "
        "Audit ONLY those two, and never pass '/' or any path outside them to a tool. "
        "Use radon_report and pyexamine_report on Python "
        "business code; use mlscent_report only if the code imports ML libraries. Read files "
        "before judging them. Work quickly: at most 8 tool calls in total. "
        "Finish by writing a concise AUDIT.md (max 25 lines) with your top findings, "
        "each with file, smell, and a one-line fix."
    ),
)
print("✅ deep agent ready — tools:", [t.__name__ for t in (radon_report, pyexamine_report, mlscent_report)])

In [ ]:
# Let it loose. Watch the todo list appear, then the tool calls.
run = deep_auditor.invoke(
    {"messages": [("user", "Audit the Python code in this directory (start with inventory.py) "
                           "and produce AUDIT.md.")]},
    config={"recursion_limit": 40},
)

for m in run["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        print(f"🔩 tool call → {tc['name']}({str(tc['args'])[:60]})")

final = run["messages"][-1].content          # reasoning models return a list of blocks
if isinstance(final, list):
    final = "\n".join(b.get("text", "") for b in final
                      if isinstance(b, dict) and b.get("type") == "text")
print("\n--- final answer ---\n", str(final)[:800])

if pathlib.Path("AUDIT.md").exists():
    print("\n📄 AUDIT.md written:\n", pathlib.Path("AUDIT.md").read_text()[:900])
cost_report()

> ⚠️ **Reality check — say this out loud:** an autonomous agent with file-write access and no gate is exactly
> the configuration you should *not* ship. Notice that nothing in section 4.3 ran `pytest`. The deep agent is
> excellent at **exploration and reporting**; Parts 1–3 are what you wrap around it before it is allowed to
> **change** anything.
>
> 🐛 **A real bug from building this notebook.** The first version of `radon_report` had no path guard and
> no length limit. The agent decided to audit `/` — and returned a **6.8 MB** tool result, stalling the run and
> burning the token budget. A fixed-flow pipeline cannot do this, because *you* choose the arguments. The moment
> an agent chooses its own, **every tool needs a bounded input and a bounded output.** That is the tax autonomy
> charges, and it is why `_safe()` and `LIMIT` exist two cells up.

## 4.4 · Under the hood: how tool-calling actually works

§4.3's deep agent "used tools". Nothing mystical happened. The model **cannot execute anything** — the loop
around it does. Here is the entire protocol:

1. Your framework converts each Python function into a **JSON schema** (name, description, parameter types) and
   sends it with the prompt. *The docstring becomes the description the model reads* — which is why the
   docstrings on `radon_report` and friends are written for a model, not a human.
2. Instead of prose, the model may emit a structured **tool call**: `{"name": "radon_report",
   "arguments": {"path": "inventory.py"}}`. That is just text in a special slot.
3. **Your code** parses it, calls the real Python function, and appends the result as a `tool` message.
4. The model is called *again*, now with the result in its context. It either calls another tool or answers.

Steps 2–4 repeat. That loop is the agent. `recursion_limit=40` is the stop condition — the same
`max_iterations` idea as Part 2, wearing a different name.

Two consequences worth stating plainly:

- **The model never touches your filesystem.** It emits a *request*; your code decides whether to honour it.
  Every safety property lives in step 3 — which is exactly where `_safe()` and `LIMIT` went.
- **Docstrings are prompt engineering.** A vague docstring produces wrong arguments. This is the rare case
  where writing better documentation immediately makes the software work better.

Let's look at the actual schema the model receives:

In [ ]:
# What deepagents/LangChain sends to the model for ONE of our tools
from langchain_core.tools import tool as make_tool

schema = make_tool(radon_report).args_schema.model_json_schema()
print("TOOL NAME :", radon_report.__name__)
print("DESCRIPTION the model reads (i.e. your docstring):")
print("   ", radon_report.__doc__.strip())
print("\nPARAMETER SCHEMA:")
print(json.dumps(schema, indent=2)[:400])
print("\n☝️ Change the docstring, and you have changed the model's behaviour.")

> 🧪 **Try it (3 min):** rewrite `radon_report`'s docstring to just `"""Reports things."""`, rebuild the deep
> agent, and re-run §4.3. Does it still pick the right tool for the right file? This is the cheapest possible
> demonstration that **docstrings are part of your program's behaviour** once an agent is reading them.

## 4.5 · `debtbuster` — the same pipeline, as a real package 📦

Notebooks are for learning. **CI runs commands.** So everything you built by hand in Parts 1–3 also exists as
a published, `pip install`-able package:

### 🌍 [`github.com/KarthikShivasankar/debtbuster`](https://github.com/KarthikShivasankar/debtbuster)

We are **not** going to paste its source into this notebook — you already wrote every idea in it by hand.
Instead: understand what it is, install it the way a CI job would, and drive it.

---

### What `debtbuster` actually does

One command takes a Python file and either **reports on it** or **rewrites it behind a gate**:

```bash
debtbuster audit  inventory.py                              # measure, don't touch
debtbuster fix    inventory.py --tests test_inventory.py    # rewrite, but only if it passes
```

`audit` is **purely deterministic — it never calls a model, so it costs nothing.** It runs radon for average
cyclomatic complexity and PyExamine for code-level smells, and prints what they found. This is the mode you
run on every commit.

`fix` is the whole Part 2 team, packaged:

```
        ┌───────────┐  smells   ┌──────────────┐ candidate ┌──────────┐
START ─▶│  auditor  │──────────▶│  refactorer  │──────────▶│  QA gate │──▶ ✅ END
        └───────────┘           └──────────────┘           └──────────┘
         PyExamine, then          an LLM, under              ast · pytest · radon
         an LLM reads it          hard invariants            ⚠️ NO LLM IN HERE
                                        ▲                         │
                                        └──── rejection reason ────┘
                                             (at most 3 iterations)
```

**The auditor** runs PyExamine over the file's directory, hands the raw findings *plus* the source to the
model, and asks for at most six named smells. Same division of labour as §1.6: the tool measures, the model
interprets what the tool measured. It is not asked to compute anything.

**The refactorer** rewrites the entire module under hard invariants baked into its system prompt — *same
public API, same behaviour, stdlib only, named constants for magic numbers, no duplication, no dead code* —
and must reply with exactly one fenced `python` block. If it replies with prose instead, the parser raises
and the loop treats that as a rejection. That is contract drift, handled.

**The QA gate** decides, and it contains no LLM on purpose:

| Gate | Tool | What it catches | Cost |
|---|---|---|---|
| 1 · syntax | `ast.parse` | truncated or mangled output | microseconds |
| 2 · behaviour | `pytest` in a throwaway sandbox | **plausible-but-wrong code** — the dangerous kind | ~1 s |
| 3 · quality | `radon` average complexity | "fixes" that made the module worse | ~0.1 s |

Cheapest gate first: fail fast, spend compute only on survivors. Gate 2 is **skipped when you omit
`--tests`** — which is exactly the ceiling test debt puts on an agent.

A rejection is not a retry. The gate's reason — the failing assertion with expected vs actual, the
`SyntaxError`, the two complexity numbers — is appended to the next prompt. Iteration 2 addresses the real
failure instead of resampling the same mistake, and that feedback comes from **a tool, never from another
model**.

---

### Why it is six files and not one

| File | Role | Contains an LLM call? |
|---|---|---|
| `config.py` | model, temperature, iteration budget | — |
| `brain.py` | the **only** function that talks to a model | ✅ |
| `tools.py` | deterministic eyes: radon, PyExamine | ❌ |
| `gates.py` | the QA gate | ❌ |
| `graph.py` | the team, as a LangGraph `StateGraph` with a conditional retry edge | — |
| `cli.py` | `audit` / `fix`, argument parsing, **and the exit code** | — |

Read that third column top to bottom. **Exactly one file can hallucinate.** You can audit the trustworthiness
of the whole package by reading `gates.py` and `tools.py` — about 60 lines — and never opening `brain.py`.
That separation is not tidiness; it is what makes the harness reviewable.

> 🔑 **The exit code is the product.** `debtbuster fix` returns **0** only when a candidate passed every gate,
> and **1** otherwise — writing nothing. That single integer is what turns an agent into something a CI
> pipeline can depend on: a green build means *verified*, not *the model sounded confident*.
>
> ```yaml
> - run: debtbuster fix src/module.py --tests tests/test_module.py
>   env:
>     OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
> ```

### Install it the way CI would 🌍

One line. No `%%writefile`, no copy-paste, no cloning — pip resolves the GitHub URL, builds the package, and
puts a `debtbuster` command on this Colab VM's `PATH`.

> ⏳ Takes ~40 s: it pulls in `langgraph`, `openai`, `radon`, `pylint` and PyExamine as declared dependencies.
> Most are already installed from §0.1, so pip mostly just checks them.

In [ ]:
%pip install -q "git+https://github.com/KarthikShivasankar/debtbuster.git"

import os, pathlib, shutil, subprocess, sys

# Colab installs console scripts onto the PATH already; a local venv may not.
# §0.5 did this once — repeat it here so this section stands on its own.
os.environ["PATH"] = str(pathlib.Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

shown = subprocess.run([sys.executable, "-m", "pip", "show", "debtbuster"],
                       capture_output=True, text=True).stdout
version = next((l.split(":", 1)[1].strip() for l in shown.splitlines()
                if l.startswith("Version:")), None)
where = shutil.which("debtbuster")

print("debtbuster version:", version or "❌ not installed")
print("on PATH           :", where or "❌ not found")
assert version and where, "install failed — re-run this cell; if it still fails, check the network"
print("\n✅ ready. The next cell drives it.")

> 🔑 **Where did your API key go?** Nowhere. It is still only in `os.environ`, put there by §0.2.
> `debtbuster` builds its client with `OpenAI()`, which reads `OPENAI_API_KEY` from the environment, and every
> `!debtbuster …` below is a **subprocess of this kernel** — so it inherits the variable automatically.
>
> Nothing is baked into the package. That is precisely why the same install is safe in CI, where the key
> arrives as a repository secret and the code never changes.

### Drive it 🚗

Three commands on the same patient you have been operating on all afternoon.

> 🎯 **Predict first:** set `MY_GUESS_EXIT` in the next cell. Will `debtbuster fix` exit **0** (a patch passed
> every gate) or **1** (the gate held)? Think about which answer you would rather see in a CI log.

In [ ]:
MY_GUESS_EXIT = 0   # 🎯 EDIT ME: what exit code will `debtbuster fix` return — 0 or 1?

# ① the interface
!debtbuster --help

print("\n" + "=" * 70)
# ② audit — deterministic only. No model, no tokens, no cost.
!debtbuster audit inventory.py

print("\n" + "=" * 70)
# ③ fix — the full auditor → refactorer ⇄ QA team, behind the three gates.
!debtbuster fix inventory.py --tests test_inventory.py

# IPython records the exit status of the last `!` command. That integer IS the CI contract.
exit_code = _exit_code
MEANING = {0: "✅ a patch passed every gate",
           1: "🛡 the gate held — nothing unverified shipped",
           127: "⚠️ debtbuster is not on PATH — re-run the install cell above"}
print(f"\nEXIT CODE = {exit_code}   {MEANING.get(exit_code, '⚠️ unexpected — this is not a gate verdict')}")
assert exit_code in (0, 1), "not a gate verdict — fix the install before reading anything into this"

quiz("Would `debtbuster fix` exit 0 (accepted) or 1 (rejected)?",
     MY_GUESS_EXIT, exit_code,
     "Either outcome is a SUCCESS for the harness. A pipeline that can only ever exit 0 is not a gate, "
     "it is a rubber stamp — and you would never learn that your model just broke behaviour. "
     "Note the assert above: an exit code you did not plan for (127 = command not found) is NOT a verdict, "
     "and treating it as one is how a broken CI job turns into a silent green build.")

### 🧪 Try it (3 min) — take the safety net away

`fix` skips gate 2 when you omit `--tests`. Run it both ways on the same file:

```bash
!debtbuster fix inventory.py                              # gates 1 and 3 only
!debtbuster fix inventory.py --tests test_inventory.py    # all three
```

Then diff the two results. Without tests, nothing is checking that the module still *does* the same thing —
only that it parses and scores well on complexity. A model that quietly drops the refund logic would sail
through, and complexity would even *improve*, because deleting code always does.

> 🧠 **The lesson, stated as a rule:** *an agent may only be trusted to change what your tests already pin
> down.* Test coverage is not a quality metric here — it is the literal boundary of what you can safely
> automate. This is why "we'll add tests later" costs more than it looks like it costs.

### Where does `debtbuster` actually live? 🤔

A fair question, since nothing here came from your laptop.

- **The source lives on GitHub** — [`github.com/KarthikShivasankar/debtbuster`](https://github.com/KarthikShivasankar/debtbuster), a public repository with its own README,
  licence and test suite. It will still be there next week.
- **The install lives on this Colab VM.** pip built the package into the VM's site-packages and registered a
  `debtbuster` console script on its `PATH`. When the runtime disconnects, *that* is gone — re-run one cell
  and it comes back.
- **We drive it with `!`, not `import debtbuster`.** A subprocess picks up a freshly installed console script
  immediately, while the already-running kernel would need a restart. Using the CLI is also the honest test:
  it is exactly how CI will invoke it.

> 📌 **The whole point of packaging.** The key lives in the *environment* and the interface is a *command*, so
> this works identically on your laptop, in this notebook, and in a GitHub Actions runner:
>
> ```bash
> pip install "git+https://github.com/KarthikShivasankar/debtbuster.git"
> debtbuster fix src/module.py --tests tests/test_module.py
> ```
>
> The notebook taught the ideas. The CLI is what ships.

## 4.6 · ✍️ Exercise 4 (pick one, ~15 min)

All three start the same way — get your own copy of the source:

```bash
!git clone https://github.com/KarthikShivasankar/debtbuster.git
%pip install -q -e ./debtbuster
```

(Or fork it on GitHub first, if you want somewhere to push the answer.)

**A · Model swap.** Change one line — `MODEL` in `debtbuster/src/debtbuster/config.py` — to `gpt-4.1` or
`gpt-4o-mini`, then re-run `debtbuster fix`. Measure three things: iterations to acceptance, wall-clock, and
cost. *Is the expensive model actually cheaper **per accepted patch**?* One line changes the brain because
every call routes through `brain.chat()` — that centralisation was the point.

**B · A fourth gate.** In `gates.py`, add a gate that runs MLScent (`ml_smell_detector`) and rejects any patch
that *increases* the ML smell count. Then ask the harder question: does anything still get accepted? What does
that tell you about where to set a quality bar?

**C · Break a gate on purpose.** Delete gate 2 (the pytest call) from `gates.py`, re-run `fix`, and diff the
result against the gated one. How much worse is the patch — and, more unsettling, *how would you have known*
without the gate?

> 🧠 **Whichever you pick, notice what you did not have to do:** no re-plumbing, no prompt archaeology. The
> package's file boundaries are drawn along *responsibilities*, so a change to the brain, the gates, or the
> tools stays inside one file. That is what "production-ready" buys you — not more features, fewer places to
> look.

In [ ]:
# 🖊️ Your Exercise 4 workspace

## 4.7 · Checkpoint ✅


1. Why does the QA gate stay identical across the hand-rolled, LangGraph and deep-agent versions?
2. What is the one-line change that repoints this whole harness at a different model or provider?
3. Which part of today's pipeline would you *never* let run unattended on a production repo, and why?

<details><summary>Answers</summary>

1. Because **verification is a property of the task, not of the framework**. Frameworks orchestrate; they do not decide what "correct" means.
2. `config.MODEL` in the package (or `MODEL_NAME` in the notebook). Everything else routes through one `chat()` / `llm()` function — that is why it was worth centralising on the very first cell.
3. The **write** step. Auditing and reporting are safe to automate; changing code is safe to automate only behind a gate *plus* human review. Agents propose; verified pipelines and humans dispose.
</details>

---
# 🎓 Wrap-up

## What you built today

| # | Agent | Tools | Verified by |
|---|---|---|---|
| 1 | Code Auditor | radon · pylint · PyExamine | JSON contract |
| 2 | ML Auditor | MLScent | JSON contract |
| 3 | Refactorer | — | the QA gates |
| 4 | QA Verifier | ast · pytest · radon | *it is the verifier* |
| 5 | TD Classifier | — | gold labels |
| 6 | Triage Agent | — | human judgement |

## Scaling this up in the real world

| Today's toy | Production equivalent |
|---|---|
| `WorkflowState` dataclass | LangGraph state graphs · AutoGen conversations · CrewAI crews |
| a prompted general model | fine-tuned small models — cheaper *and* more consistent for narrow tasks |
| zero-shot TD classifier | **BEACon-TD / TD-Suite** fine-tuned transformers (13 debt types) |
| 4 hand-wrapped tools | **PyExamine** (49 metrics) · **MLScent** (76 ML anti-patterns) · full linter farms |
| 7 pytest tests as the gate | full CI: coverage thresholds, mutation testing, canary deploys |

## Papers & tools from today

- **PyExamine** — *MSR 2025* · `pip install code-quality-analyzer` · [github.com/KarthikShivasankar/python_smells_detector](https://github.com/KarthikShivasankar/python_smells_detector)
- **MLScent** — *CAIN 2025* · `pip install ml-code-smell-detector` · [arXiv:2502.18466](https://arxiv.org/abs/2502.18466)
- **BEACon-TD / TD-Suite** — *Journal of Systems and Software, 2025* · [github.com/KarthikShivasankar/text_classification](https://github.com/KarthikShivasankar/text_classification)
- *Enhancing Python Code Maintainability through LLM-Based Approaches* — Shivashankar & Martini, 2025
- **`debtbuster`** — today's pipeline as an installable CLI · [github.com/KarthikShivasankar/debtbuster](https://github.com/KarthikShivasankar/debtbuster)
- Fowler, *Refactoring* (2nd ed.) — the smell taxonomy everything builds on
- Cunningham (1992) — the original "debt" metaphor. Two pages. Read it verbatim.

## The three sentences worth remembering

1. **Deterministic tools measure; the LLM interprets; a gate decides.**
2. **Never let the component that can hallucinate be the component that certifies it did not.**
3. **Capability grows through tools and verification, not through bigger models.**

In [ ]:
# Final bill for the whole workshop
cost_report()
scoreboard()
pit_stop("Part 4")
print("\n📁 artefacts in", os.getcwd(), ":")
for p in sorted(pathlib.Path(".").glob("*")):
    print("  ", p.name)
print("\n🎉 Thanks for building with us! — Karthik & Adela · LLMA4SE 2026")